# Overview

This notebook ingests a directory of 60-second OGG soundscape
recordings and writes `submission.csv` containing 234 per-window
probabilities. In public-commit dry-runs it additionally runs a
complete GroupKFold OOF on the labeled soundscapes (HGNet + SED
+ HTP K-fold + assembled approximation of the final blend),
scored with `macro_auc`.


# BirdCLEF+ 2026 — Pantanal acoustic species identification

**V54: HTP ablation test.** The V53 complete OOF (on labeled
soundscapes) showed HTP at every positive weight strictly hurts the
full blend macro-AUC (w=0 -> 0.9434; w=0.15 -> 0.9237). V125 = a
byte-identical V41 scored 0.943, not the canonical 0.944, putting
V41's HTP-driven 'win' inside the LB-noise band. V54 ships the
OOF-validated ablation: V41's pipeline minus the HTP post-blend, so
submission.csv is pc010 + HGNet (0.92 / 0.08), no HTP layer.

The HTP cells (Sarkar + head + training + inference + OOF report)
still run so the kernel log shows the full OOF picture; only the
HTP post-blend cell that writes HTP into submission.csv is omitted.


In [ ]:
# ── Cell 0: Install ONNX Runtime + TF 2.20 ────────────────────────────
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

# Try ONNX first (150x faster than TF SavedModel) — search anywhere under /kaggle/input
import glob as _glob
_onnx_whls = [w for w in _glob.glob("/kaggle/input/**/onnxruntime*cp312*x86_64*.whl", recursive=True) if "gpu" not in w]
if _onnx_whls:
    print(f"Installing ONNX from: {_onnx_whls[0]}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", _onnx_whls[0]], check=True)
    print("ONNX Runtime installed")
else:
    print("WARNING: No ONNX wheel found")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
print("TF 2.20 installed")

try:
    import onnxruntime as ort
    _ONNX_AVAILABLE = True
    print("ONNX Runtime available ✅")
except ImportError:
    _ONNX_AVAILABLE = False
    print("ONNX not available, falling back to TF")

In [ ]:
# ── Cell 1: Mode switch ────────────────────────────────────────────────
MODE = "submit"   # ← change to "train" for local CV
 
assert MODE in {"train", "submit"}
print("MODE =", MODE)

## Configuration

The notebook is configured for Kaggle code-competition inference. In submit mode it skips expensive OOF reporting and focuses on producing `submission.csv` under the hidden `test_soundscapes` mount. When hidden test files are absent, the notebook runs a dry-run on train soundscape files so we can validate shape, NaNs, and timing.

Two details matter for reproducibility:

- `enable_internet` is disabled in kernel metadata, so every dependency must be attached as a Kaggle input.
- The pipeline runs on CPU. ONNX Runtime is used because it is much faster and more stable for this Perch inference path than a full TensorFlow-only route.


In [ ]:
# ── Cell 2: Imports & config ───────────────────────────────────────────
import os, re, gc, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
 
tf.experimental.numpy.experimental_enable_numpy_behavior()
try: tf.config.set_visible_devices([], "GPU")
except: pass
 
_WALL_START = time.time()
 
BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
WORK_DIR  = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)
 
SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12          # 12 × 5s = 60s per file
 
CFG = {
    # inference
    "batch_files": 16,

    # local CV
    "oof_n_splits": 5   if MODE == "train" else 3,

    # dry-run
    "dryrun_n_files": 20 if MODE == "train" else 0,

    # train-only flags
    "run_oof": True,  # V54: required for run_pipeline_oof -> oof_pipeline
    "verbose": MODE == "train",

    # V18 proto_ssm
    "proto_ssm_train": {
        "n_epochs":        80  if MODE == "train" else 40,
        "lr":              8e-4,
        "weight_decay":    1e-3,
        "val_ratio":       0.15,
        "patience":        20  if MODE == "train" else 8,
        "pos_weight_cap":  25.0,
        "distill_weight":  0.15,
        "proto_margin":    0.15,
        "label_smoothing": 0.03,
        "oof_n_splits":    5   if MODE == "train" else 3,
        "mixup_alpha":     0.4,
        "focal_gamma":     2.5,
        "swa_start_frac":  0.65,
        "swa_lr":          4e-4,
        "use_cosine_restart": True,
        "restart_period":  20,
    },
    "residual_ssm": {
        "d_model": 128, "d_state": 16, "n_ssm_layers": 2,
        "dropout": 0.1, "correction_weight": 0.35,
        "n_epochs": 40  if MODE == "train" else 20,
        "lr": 8e-4,
        "patience": 12  if MODE == "train" else 6,
    },
    "mlp_params": {
        "hidden_layer_sizes": (256, 128), "activation": "relu",
        "max_iter": 500  if MODE == "train" else 200,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20  if MODE == "train" else 10,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    },
}
print("✅ V18 CFG loaded")
print(f"  n_epochs={CFG['proto_ssm_train']['n_epochs']}  "
      f"patience={CFG['proto_ssm_train']['patience']}  "
      f"oof_n_splits={CFG['proto_ssm_train']['oof_n_splits']}  "
      f"mlp_max_iter={CFG['mlp_params']['max_iter']}")
 
print("Config ready")
print(f"  run_oof={CFG['run_oof']}  verbose={CFG['verbose']}  dryrun={CFG['dryrun_n_files']}")

## Data and Label Setup

The competition target has 234 scored classes. The notebook reads `taxonomy.csv` and `sample_submission.csv`, then constructs the row/column order required by Kaggle. It also parses train soundscape labels for the internal dry-run and for building sequence-model training targets.

The hidden test set is not available during ordinary public execution. This is expected for a Kaggle code competition: the same notebook will see real `test_soundscapes` only when Kaggle reruns it as a formal competition submission.


In [ ]:
# ── Cell 3: Data loading & label parsing ──────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
 
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}
 
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
 
def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}
 
def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)
 
sc = (soundscape_labels
      .groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels)
      .reset_index(name="label_list"))
 
sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
 
_meta = sc["filename"].apply(parse_fname).apply(pd.Series)
sc = pd.concat([sc, _meta], axis=1)
 
Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1
 
windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)
 
full_rows = (sc[sc["fully_labeled"]]
             .sort_values(["filename", "end_sec"])
             .reset_index(drop=False))
Y_FULL = Y_SC[full_rows["index"].to_numpy()]
 
print(f"Classes: {N_CLASSES} | Fully-labeled files: {len(full_files)}")
print(f"Full-file windows: {len(full_rows)} | Active classes: {int((Y_FULL.sum(0) > 0).sum())}")

## Perch Backbone

Perch provides strong acoustic embeddings and class logits. The notebook prefers an attached ONNX export for speed, while still keeping the Kaggle TensorFlow Perch model available. In our submitted run, ONNX Perch was used and the train cache was built in roughly 2.5 minutes.

A useful implementation detail is the species mapping step. Most target classes map directly to Perch logits; a few unmapped targets can borrow genus-level proxy signal; the remaining unmapped species are handled by the downstream learned and prior components rather than by direct Perch logits.


In [ ]:
# ── Cell 4: Load Perch model (ONNX preferred) ─────────────────────────
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn       = birdclassifier.signatures["serving_default"]

# ONNX session (150x faster)
import glob as _glob
_perch_onnx = _glob.glob("/kaggle/input/**/perch_v2*.onnx", recursive=True)
ONNX_PERCH_PATH = Path(_perch_onnx[0]) if _perch_onnx else Path("/kaggle/input/perch-onnx-for-birdclef-2026/perch_v2.onnx")
print(f"Perch ONNX: {ONNX_PERCH_PATH} (exists={ONNX_PERCH_PATH.exists()})")
USE_ONNX = _ONNX_AVAILABLE and ONNX_PERCH_PATH.exists()

if USE_ONNX:
    _so = ort.SessionOptions()
    _so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("Using ONNX Perch (150x faster)")
else:
    print("Using TF SavedModel Perch")

bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)

mapping = (taxonomy
           .merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                  on="scientific_name", how="left"))
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")

In [ ]:
# ── Cell 4b: Genus proxy logits for unmapped species ──────────────────
import re as _re

# Find which species have no direct Perch mapping
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

# For each unmapped species, find genus-level matches in Perch vocab
proxy_map = {}   # label_idx -> list of bc_indices

unmapped_df = (taxonomy[taxonomy["primary_label"]
               .isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])]
               .copy())

for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci    = str(row["scientific_name"])
    genus  = sci.split()[0]
    
    # Find all Perch labels from the same genus
    hits = bc_labels[
        bc_labels["scientific_name"]
        .astype(str)
        .str.match(rf"^{_re.escape(genus)}\s", na=False)
    ]
    
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

# Only use proxies for biologically meaningful taxa
PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map  = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

print(f"Unmapped species total:        {len(UNMAPPED_POS)}")
print(f"Species with genus proxy:      {len(proxy_map)}")
print(f"Species still without signal:  {len(UNMAPPED_POS) - len(proxy_map)}")
print("\nProxy targets:")
for idx, bc_idxs in list(proxy_map.items())[:8]:
    label = PRIMARY_LABELS[idx]
    cls   = CLASS_NAME_MAP.get(label, "?")
    print(f"  {label:12s} ({cls:10s}) ← {len(bc_idxs)} Perch genus matches")

## Window-Level Inference Cache

The inference engine splits each 60-second soundscape into twelve 5-second windows. Caching the Perch scores and 1536-dimensional embeddings is important because later cells train several lightweight models over the same windows.

The cache also makes the notebook easier to debug: if a later modeling cell changes, the expensive audio pass does not have to be repeated inside the same run.


In [ ]:
# ── Cell 5: Perch inference engine (ONNX + multithreaded I/O) ─────────
import concurrent.futures

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:                      y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536),      dtype=np.float32)

    wr  = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        # Prefetch first batch
        next_paths   = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

        for start in itr:
            batch_paths  = next_paths
            batch_n      = len(batch_paths)
            batch_audio  = [f.result() for f in future_audio]

            # Prefetch next batch immediately
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths   = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]

            x  = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr

            for bi, path in enumerate(batch_paths):
                y    = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids  [wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites    [wr:wr + N_WINDOWS] = meta["site"]
                hours    [wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS

            # ── ONNX or TF inference ───────────────────────────────────
            if USE_ONNX:
                outs   = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
                logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
                emb    = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            else:
                out    = infer_fn(inputs=tf.convert_to_tensor(x))
                logits = out["label"].numpy().astype(np.float32)
                emb    = out["embedding"].numpy().astype(np.float32)

            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs  [br:wr]             = emb

            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)

            del x, logits, emb, batch_audio
            gc.collect()

    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames,
                             "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

print("✅ Perch inference engine (ONNX + multithreaded I/O) defined")

In [ ]:
# ── Cell 6: Build-or-load Perch training cache ────────────────────────
print(f"USE_ONNX = {USE_ONNX}  "
      f"(cache will be built with {'ONNX' if USE_ONNX else 'TF SavedModel'})")

# Add any external cache locations here if you want to reuse pre-built data
EXTERNAL_CACHE_DIRS = [
    Path("/kaggle/input/notebooks/vyankteshdwivedi/notebook1b25083f0d"),
    Path("/kaggle/input/datasets/jaejohn/perch-meta"),
]

CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL  = WORK_DIR / "perch_arrays.npz"


def _find_external_cache():
    for d in EXTERNAL_CACHE_DIRS:
        meta = d / "perch_meta.parquet"
        npz  = d / "perch_arrays.npz"
        if meta.exists() and npz.exists():
            return meta, npz
    return None, None


SCORE_KEYS = ["scores", "sc", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS   = ["embs", "emb", "embeddings", "features", "perch_embs", "arr_1"]


def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k

    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k

    raise KeyError(f"None of {candidates} found in npz. Available keys: {arr.files}")


def _build_cache():
    print(f"Building Perch cache from {len(full_files)} training files…")

    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]

    t0 = time.time()

    meta_built, sc_built, emb_built = run_perch(
        train_paths,
        batch_files=CFG["batch_files"],
        verbose=True
    )

    print(f"  Perch pass done in {time.time()-t0:.1f}s  "
          f"scores={sc_built.shape} embs={emb_built.shape}")

    meta_built.to_parquet(CACHE_META_LOCAL)

    np.savez(
        CACHE_NPZ_LOCAL,
        scores=sc_built.astype(np.float32),
        embs=emb_built.astype(np.float32),
        primary_labels=np.array(PRIMARY_LABELS)
    )

    print(f"  Cache saved to {WORK_DIR}")

    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL


ext_meta, ext_npz = _find_external_cache()

if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
    print(f"Using external cache: {CACHE_META.parent}")

elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
    print(f"Using local cache: {WORK_DIR}")

else:
    print("No cache found — building from scratch (~1.5 min)")
    CACHE_META, CACHE_NPZ = _build_cache()


print("Loading Perch cache from:", CACHE_META.parent)

meta_tr = pd.read_parquet(CACHE_META)
_arr    = np.load(CACHE_NPZ)


sc_tr_raw,  sk = _pick_array(_arr, SCORE_KEYS, N_CLASSES)
emb_tr_raw, ek = _pick_array(_arr, EMB_KEYS,   1536)

print(f"  scores ← '{sk}'  shape={sc_tr_raw.shape}")
print(f"  embs   ← '{ek}'  shape={emb_tr_raw.shape}")


sc_tr  = sc_tr_raw.astype(np.float32)
emb_tr = emb_tr_raw.astype(np.float32)


if "primary_labels" in _arr.files:
    if _arr["primary_labels"].tolist() != PRIMARY_LABELS:
        print("  WARNING: cached primary_labels differ — scores columns may not align!")
    else:
        print("  primary_labels schema OK")


if "row_id" not in meta_tr.columns:
    print("  row_id missing — reconstructing")

    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)

    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5

    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)

    meta_tr["row_id"] = (
        meta_tr["filename"].str.replace(".ogg", "", regex=False)
        + "_" + end_sec.astype(str)
    )


row_id_to_index = full_rows.set_index("row_id")["index"]

missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)

if missing_rows:
    raise RuntimeError(
        f"Cache has {len(missing_rows)} row_ids not in labeled set. "
        f"Delete {CACHE_META_LOCAL} and {CACHE_NPZ_LOCAL} to rebuild."
    )


Y_FULL_aligned = Y_SC[
    row_id_to_index.loc[meta_tr["row_id"]].to_numpy()
]

print(f"sc_tr: {sc_tr.shape}  emb_tr: {emb_tr.shape}  Y_FULL_aligned: {Y_FULL_aligned.shape}")

## Validation and Post-Processing Helpers

The local dry-run metric is useful for finding broken outputs, but it is not a replacement for public LB. Several SED-only candidates scored extremely high on train-soundscape dry-runs and then underperformed on the official public LB. For this reason, the dry-run is treated as a sanity check plus ranking clue, not as proof.

The post-processing stack combines:

- temporal smoothing across adjacent 5-second windows,
- site/hour priors from train soundscape metadata,
- file-level confidence scaling,
- per-taxon temperature scaling,
- rank-aware scaling,
- adaptive smoothing that reacts to prediction deltas.


In [ ]:
# ── Cell 7: Metric helpers ─────────────────────────────────────────────
def macro_auc(y_true, y_score):
    """
    Exact replica of the competition metric:
    macro-averaged ROC-AUC, skipping classes with no positive labels.
    This is the ONLY number you should track locally.
    """
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")
 
 
def honest_oof_auc(scores, Y, meta_df, n_splits=5, label="scores"):
    """
    GroupKFold by filename — files never split across folds.
    This is the only correct way to estimate LB performance locally.
    Leaking a file across train/val inflates AUC by ~0.01–0.03.
    """
    groups = meta_df["filename"].to_numpy()
    gkf    = GroupKFold(n_splits=n_splits)
    oof    = np.zeros_like(scores, dtype=np.float32)
 
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(scores, groups=groups), 1):
        oof[va_idx] = scores[va_idx]
 
    auc = macro_auc(Y, oof)
    print(f"[{label}] honest OOF macro-AUC: {auc:.6f}")
    return auc, oof

In [ ]:
# ── Cell 7b: Temporal smoothing helper ─────────────────────────────────
def smooth_predictions(probs, n_windows=12, alpha=0.3):
    """
    For each file's 12 windows, blend each window with its neighbors.
    
    new[t] = (1 - alpha) * old[t] + 0.5*alpha * (old[t-1] + old[t+1])
    
    alpha=0: no smoothing (your current baseline)
    alpha=0.3: moderate smoothing (good starting point)
    
    Shape: (n_files * 12, n_classes) → same shape output
    """
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"
    
    # Reshape to (n_files, 12, 234) so we can work file-by-file
    view = probs.reshape(-1, n_windows, C).copy()
    
    # Shift left and right (with edge padding = repeat boundary)
    prev_w = np.concatenate([view[:, :1, :],  view[:, :-1, :]], axis=1)  # t-1
    next_w = np.concatenate([view[:, 1:,  :], view[:, -1:, :]], axis=1)  # t+1
    
    smoothed = (1 - alpha) * view + 0.5 * alpha * (prev_w + next_w)
    
    return smoothed.reshape(N, C)


print("✅ Temporal smoothing helper defined")

In [ ]:
# ── Cell 7c: Prior table builder ───────────────────────────────────────
def build_prior_tables(sc_df, Y_labels):
    """
    Build site-level and hour-level species frequency tables.
    
    These answer: "How often is species X observed at site S at hour H?"
    
    We use these as a soft prior: add them to raw Perch logits.
    """
    sc_df = sc_df.reset_index(drop=True)
    global_p = Y_labels.mean(axis=0).astype(np.float32)  # overall frequency
    
    # ── Site-level frequencies ──────────────────────────────────────────
    site_keys = sorted(sc_df["site"].dropna().astype(str).unique())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_p    = np.zeros((len(site_keys), Y_labels.shape[1]), dtype=np.float32)
    site_n    = np.zeros(len(site_keys), dtype=np.float32)
    
    for s in site_keys:
        i     = site_to_i[s]
        mask  = sc_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_labels[mask].mean(axis=0)
    
    # ── Hour-level frequencies ──────────────────────────────────────────
    hour_keys = sorted(sc_df["hour_utc"].dropna().astype(int).unique())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_p    = np.zeros((len(hour_keys), Y_labels.shape[1]), dtype=np.float32)
    hour_n    = np.zeros(len(hour_keys), dtype=np.float32)
    
    for h in hour_keys:
        i     = hour_to_i[h]
        mask  = sc_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_labels[mask].mean(axis=0)
    
    return {
        "global_p": global_p,
        "site_to_i": site_to_i, "site_p": site_p, "site_n": site_n,
        "hour_to_i": hour_to_i, "hour_p": hour_p, "hour_n": hour_n,
    }


def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    """
    Add a scaled prior logit to the raw Perch scores.
    
    lambda_prior=0: no effect (your baseline)
    lambda_prior=0.4: moderate influence from location/time
    
    The prior is converted to a logit (log-odds) before adding.
    This is mathematically correct — you add logits, not probabilities.
    """
    eps = 1e-4
    n   = len(scores)
    out = scores.copy()
    
    # Start from global average
    p = np.tile(tables["global_p"], (n, 1))  # (n, 234)
    
    # Override with hour-level estimate (if enough data)
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j   = tables["hour_to_i"][h]
            nh  = tables["hour_n"][j]
            w   = nh / (nh + 8.0)   # shrink toward global if little data
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]
    
    # Override with site-level estimate (if enough data)
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j   = tables["site_to_i"][s]
            ns  = tables["site_n"][j]
            w   = ns / (ns + 8.0)   # same shrinkage logic
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]
    
    # Convert prior probability to logit and add
    p      = np.clip(p, eps, 1 - eps)
    logit_prior = np.log(p) - np.log1p(-p)
    out   += lambda_prior * logit_prior
    
    return out.astype(np.float32)


print("✅ Prior table functions defined")

In [ ]:
# ── Cell 7d: File-level confidence scaling ─────────────────────────────
def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
    """
    Scale each window's predictions by how confident the file is overall.
    
    Steps:
    1. For each file, find the top-k highest scores across all 12 windows
    2. Compute their mean → "file confidence"
    3. Multiply every window's scores by (file_confidence ** power)
    
    power=0: no effect (baseline)
    power=0.4: moderate suppression of uncertain files
    
    Why top-k and not max?
    Max is noisy (one lucky spike). Top-2 mean is more robust.
    """
    N, C = probs.shape
    assert N % n_windows == 0
    
    view      = probs.reshape(-1, n_windows, C)       # (n_files, 12, 234)
    sorted_v  = np.sort(view, axis=1)                 # sort across time
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)  # (n_files, 1, 234)
    
    scale  = np.power(top_k_mean, power)              # (n_files, 1, 234)
    scaled = view * scale                             # broadcast across 12 windows
    
    return scaled.reshape(N, C)


print("✅ File-level confidence scaling defined")

In [ ]:
# ── Cell 7e: Per-taxon temperature scaling ─────────────────────────────
# Build lookup: which species class are they?
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}   # continuous callers

# Build per-class temperature vector
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    if cls in TEXTURE_TAXA:
        temperatures[ci] = 0.95   # frogs/insects: slightly sharper
    else:
        temperatures[ci] = 1.10   # birds: slightly softer

n_texture = (temperatures < 1.0).sum()
n_event   = (temperatures > 1.0).sum()
print(f"✅ Temperatures: {n_event} event species (T=1.10), {n_texture} texture species (T=0.95)")

## Lightweight Perch-Embedding Learners

The notebook trains small models inside the inference notebook rather than relying on private checkpoints. This keeps the solution self-contained and public-input compliant.

The MLP probe branch uses PCA-compressed Perch embeddings and only trains classes with enough positive windows. The calibration block then uses isotonic calibration and class-level threshold estimates to keep probabilities better behaved before the final blend.


In [ ]:
# ── Cell 7f: UPGRADED MLP probe on PCA embeddings ─────────────────────
# CHANGE 1: Larger hidden layers (128,64), PCA 64-dim, max_iter=300
# Expected gain: +0.003–0.006 vs baseline (32,) hidden layer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

def build_class_freq_weights(Y, cap=10.0):
    total     = Y.shape[0]
    pos_count = Y.sum(axis=0).astype(np.float32) + 1.0
    freq      = pos_count / total
    weights   = 1.0 / (freq ** 0.5)
    weights   = np.clip(weights, 1.0, cap)
    weights   = weights / weights.mean()
    return weights.astype(np.float32)


def build_sequential_features(scores_col, n_windows=12):
    N = len(scores_col)
    assert N % n_windows == 0
    x     = scores_col.reshape(-1, n_windows)
    prev  = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    next_ = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean  = np.repeat(x.mean(axis=1), n_windows)
    max_  = np.repeat(x.max(axis=1),  n_windows)
    std   = np.repeat(x.std(axis=1),  n_windows)
    return prev.reshape(-1), next_.reshape(-1), mean, max_, std


def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=64, alpha_blend=0.4):
    """
    CHANGE 1: Upgraded MLP probe.
    - pca_dim: 32 → 64  (more embedding information)
    - hidden:  (32,) → (128, 64)  (more capacity)
    - max_iter: 100 → 300  (longer training)
    - min_pos: 8 → 5  (catches more rare species)
    """
    # Step 1: Compress embeddings
    scaler = StandardScaler()
    emb_s  = scaler.fit_transform(emb)
    pca    = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z      = pca.fit_transform(emb_s).astype(np.float32)
    print(f"Embedding: {emb.shape} → PCA: {Z.shape}  "
          f"(variance retained: {pca.explained_variance_ratio_.sum():.2%})")

    class_weights = build_class_freq_weights(Y, cap=10.0)

    probe_models = {}
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    print(f"Training MLP probes for {len(active)} species (>= {min_pos} pos windows)...")

    MAX_ROWS = 3000   # slightly higher budget for (128,64) layers

    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue

        prev, next_, mean, max_, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([
            Z,
            scores_raw[:, ci:ci+1],
            prev[:, None], next_[:, None],
            mean[:, None], max_[:, None], std[:, None],
        ])

        n_pos = int(y.sum()); n_neg = len(y) - n_pos
        pos_idx = np.where(y == 1)[0]

        w      = float(class_weights[ci])
        repeat = max(1, int(round(w * n_neg / max(n_pos, 1))))
        repeat = min(repeat, 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))

        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])

        clf = MLPClassifier(
            hidden_layer_sizes=(128, 64),   # CHANGE 1: was (32,)
            activation="relu",
            max_iter=300,                   # CHANGE 1: was 100
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=15,            # CHANGE 1: was 10
            random_state=42,
            learning_rate_init=5e-4,        # CHANGE 1: was 1e-3 (lower lr for deeper net)
            alpha=0.005,                    # CHANGE 1: was 0.01
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    print(f"Trained {len(probe_models)} MLP probes")
    return probe_models, scaler, pca, alpha_blend


def apply_mlp_probes(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4):
    emb_s  = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)
    result = scores_test.copy()
    for ci, clf in probe_models.items():
        prev, next_, mean, max_, std = build_sequential_features(scores_test[:, ci])
        X_test = np.hstack([
            Z_test, scores_test[:, ci:ci+1],
            prev[:, None], next_[:, None],
            mean[:, None], max_[:, None], std[:, None],
        ])
        prob  = clf.predict_proba(X_test)[:, 1].astype(np.float32)
        logit = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        result[:, ci] = (1 - alpha_blend) * scores_test[:, ci] + alpha_blend * logit
    return result

print("✅ CHANGE 1: Upgraded MLP probe (pca_dim=64, hidden=(128,64), max_iter=300, min_pos=5)")


In [ ]:
# ── Cell 7f-2: Vectorized MLP probe inference ──────────────────────────
import torch
import torch.nn as nn

class VectorizedMLPProbes(nn.Module):
    """Stacks all per-class MLP weights into a single batched PyTorch model.
    Replaces the slow Python for-loop over probe_models at inference time."""
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        V = len(self.valid_classes)
        if V == 0:
            self.weights = nn.ParameterList()
            self.biases  = nn.ParameterList()
            self.n_layers = 0
            return

        sample = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights  = nn.ParameterList()
        self.biases   = nn.ParameterList()

        for layer_idx in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[layer_idx]
                          for c in self.valid_classes], axis=0)       # (V, in, out)
            b = np.stack([probe_models[c].intercepts_[layer_idx]
                          for c in self.valid_classes], axis=0)       # (V, out)
            self.weights.append(nn.Parameter(
                torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(
                torch.tensor(b, dtype=torch.float32), requires_grad=False))

    def forward(self, x):
        # x: (V, N, in_dim)
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)   # (V, N)


def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models,
                                 scaler, pca, alpha_blend=0.4):
    """
    Drop-in replacement for apply_mlp_probes().
    Uses batched PyTorch matrix multiply instead of a Python for-loop —
    ~10-50x faster at inference time.
    """
    if len(probe_models) == 0:
        return scores_test.copy()

    emb_s  = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)

    valid_classes = sorted(probe_models.keys())
    V = len(valid_classes)
    N = len(scores_test)

    # Build sequential features for all classes at once
    raw  = scores_test[:, valid_classes].T          # (V, N)
    n_files = N // N_WINDOWS
    raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1)
    mx   = np.repeat(raw_view.max(axis=2),  N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)

    # scalar_feats: (V, N, 6)
    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)

    # Z_test: (N, D) → broadcast to (V, N, D)
    Z_expanded = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))

    # X_all: (V, N, D+6)
    X_all = np.concatenate(
        [Z_expanded.astype(np.float32), scalar_feats], axis=-1)

    vec_probe = VectorizedMLPProbes(probe_models)
    vec_probe.eval()
    with torch.no_grad():
        preds = vec_probe(torch.tensor(X_all)).numpy()   # (V, N)

    result = scores_test.copy()
    base_valid = scores_test[:, valid_classes]           # (N, V)
    result[:, valid_classes] = (
        (1.0 - alpha_blend) * base_valid +
        alpha_blend * preds.T
    )
    return result

print("✅ Vectorized MLP probe inference defined")

In [ ]:
# ── Cell 7f-3: Isotonic Calibration + Per-Class Threshold Optimization ──
# CHANGE 2: Used by top notebooks (a.txt/d.txt), expected +0.004–0.008
# Trains isotonic regression per class on OOF scores to calibrate probs,
# then finds the best F1-threshold per species via grid search.
from sklearn.isotonic import IsotonicRegression

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, 
                                       threshold_grid=None, n_windows=12):
    """
    CHANGE 2: For each species:
    1. Fit isotonic regression on OOF scores (calibrates overconfident/underconfident classes)
    2. Grid-search F1-optimal threshold over calibrated probs
    Returns: thresholds array of shape (n_classes,)
    """
    if threshold_grid is None:
        threshold_grid = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    
    n_samples, n_cls = oof_probs.shape
    thresholds = np.full(n_cls, 0.5, dtype=np.float32)
    n_files    = n_samples // n_windows
    file_oof   = oof_probs.reshape(n_files, n_windows, n_cls).max(axis=1)
    file_y     = Y_FULL.reshape(n_files, n_windows, n_cls).max(axis=1)
    
    n_calibrated = 0
    for c in range(n_cls):
        y_true = file_y[:, c]
        y_prob = file_oof[:, c]
        if y_true.sum() < 3:
            continue
        try:
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(y_prob, y_true)
            y_cal = ir.transform(y_prob)
        except Exception:
            y_cal = y_prob
        
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp = ((pred==1) & (y_true==1)).sum()
            fp = ((pred==1) & (y_true==0)).sum()
            fn = ((pred==0) & (y_true==1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec  = tp / (tp + fn + 1e-8)
            f1   = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t
        n_calibrated += 1
    
    print(f"Calibrated {n_calibrated} classes")
    print(f"Mean threshold: {thresholds.mean():.3f}")
    print(f"Range: [{thresholds.min():.2f}, {thresholds.max():.2f}]")
    return thresholds


def apply_per_class_thresholds(scores, thresholds):
    """
    Sharpens probabilities around the per-class threshold:
    - above threshold → push toward 1
    - below threshold → push toward 0
    """
    C = scores.shape[1]
    assert C == len(thresholds)
    scaled = np.copy(scores)
    for c in range(C):
        t = thresholds[c]
        above = scores[:, c] > t
        scaled[ above, c] = 0.5 + 0.5 * (scores[ above, c] - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)

print("✅ CHANGE 2: Isotonic calibration + per-class threshold optimization defined")


In [ ]:
# ── Cell 7g: Rank-aware scaling ────────────────────────────────────────
def rank_aware_scaling(probs, n_windows=12, power=0.4):
    """
    CHANGE 6: Scale each window by the file's single peak confidence.

    How it works:
      1. For each file, find the MAX score across all 12 windows (per species)
      2. Raise it to power → scale factor
      3. Multiply every window's score by that scale factor

    Example for one species across 12 windows:
      Confident file:  max=0.90 → scale=0.90^0.4=0.96 → mild boost
      Uncertain file:  max=0.10 → scale=0.10^0.4=0.40 → strong suppression

    How this differs from Change 3 (file_confidence_scale):
      Change 3 uses top-2 MEAN → smoother, less aggressive
      Change 6 uses single MAX  → asks "does ANY window have strong evidence?"

    power=0.0 → no effect (baseline)
    power=0.4 → moderate suppression of uncertain files (recommended start)
    power=1.0 → multiply directly by file max (very aggressive)
    """
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"

    view     = probs.reshape(-1, n_windows, C)              # (n_files, 12, 234)
    file_max = view.max(axis=1, keepdims=True)              # (n_files, 1, 234)

    scale  = np.power(file_max, power)                      # (n_files, 1, 234)
    scaled = view * scale                                   # broadcast to all 12 windows

    return scaled.reshape(N, C)


print("✅ Rank-aware scaling defined")

In [ ]:
# ── Cell 7h: Adaptive delta smoothing ─────────────────────────────────
def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    """
    CHANGE 7: Smooth uncertain windows toward their neighbors,
    while leaving confident windows almost untouched.

    How it works:
      For each window t:
        conf  = max probability across all 234 species at window t
        alpha = base_alpha * (1 - conf)   ← KEY: adapts to confidence
        new[t] = (1 - alpha) * old[t] + alpha * avg(old[t-1], old[t+1])

    Why alpha adapts to confidence:
      Confident window (max=0.90):
        alpha = 0.20 * (1 - 0.90) = 0.02  → barely smoothed, peak preserved
      Uncertain window (max=0.10):
        alpha = 0.20 * (1 - 0.10) = 0.18  → smoothed more, noise reduced

    This is exactly why your Change 1 hurt (-0.005) but this one should help:
      Change 1 used fixed alpha=0.3 → diluted confident peaks equally
      Change 7 uses adaptive alpha  → protects confident peaks, smooths noise

    base_alpha=0.0  → no smoothing (baseline)
    base_alpha=0.20 → recommended starting point
    """
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"

    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)    # (n_files, 12, 234) original
    out    = result.reshape(-1, n_windows, C)   # (n_files, 12, 234) to modify

    for t in range(n_windows):

        # Confidence at this window = max prob across all species
        # Shape: (n_files, 1) — one confidence value per file per window
        conf = view[:, t, :].max(axis=-1, keepdims=True)   # (n_files, 1)

        # Adaptive alpha — low confidence = more smoothing
        alpha = base_alpha * (1.0 - conf)                  # (n_files, 1)

        # Neighbor average with edge padding
        if t == 0:
            # First window: left neighbor = itself
            neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            # Last window: right neighbor = itself
            neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0

        # Blend: confident windows barely change, uncertain ones smooth more
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg

    return result


print("✅ Adaptive delta smoothing defined")

## Sequence Modeling: ProtoSSM and ResidualSSM

The strongest branch is the sequence model over Perch embeddings and window predictions. A soundscape is not just twelve unrelated clips: calls can persist, repeat, or appear in bursts. Modeling the temporal sequence helps the notebook distinguish isolated noise from consistent evidence.

The main branch is a lightweight ProtoSSM with cross-attention. A second ResidualSSM learns a correction against remaining structured error. Both are trained inside the notebook from the attached competition data, so there is no hidden checkpoint dependency.


In [ ]:
# ── Cell 7i: LightProtoSSM WITH Cross-Attention ────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model
        )
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(B_sz, D, self.d_state)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=128, d_state=16,
                 n_classes=234, n_windows=12, dropout=0.15,
                 n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=2):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc  = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        self.ssm_fwd  = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd  = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge= nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(2)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop     = nn.Dropout(dropout)

        if use_cross_attn:
            self.cross_attn = nn.ModuleList([
                nn.MultiheadAttention(d_model, num_heads=cross_attn_heads,
                                      dropout=dropout, batch_first=True)
                for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])

        self.prototypes   = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp   = nn.Parameter(torch.tensor(5.0))
        self.class_bias   = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

    def init_prototypes(self, emb_tensor, labels_tensor):
        with torch.no_grad():
            h = self.input_proj(emb_tensor)
            for c in range(self.n_classes):
                mask = labels_tensor[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]

        for i, (fwd, bwd, merge, norm) in enumerate(zip(
                self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            h_f = fwd(h); h_b = bwd(h.flip(1)).flip(1)
            h   = self.drop(merge(torch.cat([h_f, h_b], dim=-1)))
            h   = norm(h + res)
            if self.use_cross_attn:
                attn_out, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](h + attn_out)

        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = (torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp)
               + self.class_bias[None, None, :])
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            out   = alpha * sim + (1 - alpha) * perch_logits
        else:
            out = sim
        return out

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def train_light_proto_ssm(emb_full, scores_full, Y_full, meta_full,
                           n_epochs=40, patience=8, lr=1e-3,
                           n_sites=20, verbose=False):
    """Train LightProtoSSM with cross-attention + SWA."""
    n_files = len(emb_full) // N_WINDOWS
    emb_f   = emb_full.reshape(n_files, N_WINDOWS, -1)
    log_f   = scores_full.reshape(n_files, N_WINDOWS, -1)
    lab_f   = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    fnames  = meta_full["filename"].unique()
    sites_u = sorted(meta_full["site"].unique())
    site2i  = {s: i + 1 for i, s in enumerate(sites_u)}

    site_ids = np.array([
        min(site2i.get(meta_full.loc[meta_full["filename"]==fn,"site"].iloc[0], 0), n_sites-1)
        for fn in fnames], dtype=np.int64)
    hour_ids = np.array([
        int(meta_full.loc[meta_full["filename"]==fn,"hour_utc"].iloc[0]) % 24
        for fn in fnames], dtype=np.int64)

    model = LightProtoSSM(n_classes=N_CLASSES, n_sites=n_sites,
                          use_cross_attn=True, cross_attn_heads=2)
    model.init_prototypes(
        torch.tensor(emb_full, dtype=torch.float32),
        torch.tensor(Y_full,   dtype=torch.float32))
    print(f"LightProtoSSM params: {model.count_parameters():,}")

    emb_t  = torch.tensor(emb_f,    dtype=torch.float32)
    log_t  = torch.tensor(log_f,    dtype=torch.float32)
    lab_t  = torch.tensor(lab_f,    dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos_cnt    = lab_t.sum(dim=(0, 1))
    total      = lab_t.shape[0] * lab_t.shape[1]
    pos_weight = ((total - pos_cnt) / (pos_cnt + 1)).clamp(max=25.0)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")

    best_loss, best_state, wait = float("inf"), None, 0

    # ── SWA setup ──────────────────────────────────────────────────────
    swa_model = torch.optim.swa_utils.AveragedModel(model)
    swa_start = int(n_epochs * 0.65)
    swa_sched = torch.optim.swa_utils.SWALR(opt, swa_lr=4e-4)

    for ep in range(n_epochs):
        model.train()
        out  = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
        loss = (F.binary_cross_entropy_with_logits(
                    out, lab_t, pos_weight=pos_weight[None, None, :])
                + 0.15 * F.mse_loss(out, log_t))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        # ── SWA update ─────────────────────────────────────────────────
        if ep >= swa_start:
            swa_model.update_parameters(model)
            swa_sched.step()
        else:
            sched.step()

        if loss.item() < best_loss:
            best_loss  = loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    # ── Use SWA model if we reached swa_start, else best checkpoint ────
    if ep >= swa_start:
        torch.optim.swa_utils.update_bn(emb_t.unsqueeze(0), swa_model)
        model = swa_model
    else:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        out = model(emb_t, log_t, site_ids=site_t, hours=hour_t)
    print(f"LightProtoSSM trained — best loss={best_loss:.4f}")
    return model, site2i


print("✅ CHANGE 4: LightProtoSSM with cross-attention (2 heads) + SWA defined")

In [ ]:
# ── Cell 7i-2: TTA — Circular Shift Test-Time Augmentation ───────────
# CHANGE 3: Average ProtoSSM predictions across 5 time shifts
# Expected gain: +0.003–0.005 on public LB

def run_tta_proto(proto_model, emb_files, sc_files,
                  site_t, hour_t, shifts=[0, 1, -1, 2, -2]):
    """
    CHANGE 3: TTA by circular-shifting 12-window sequences.
    
    For each shift s:
      1. Roll embeddings and perch logits by s windows
      2. Run ProtoSSM → get predictions
      3. Roll predictions back by -s (undo shift)
    
    Finally average all predictions across shifts.
    
    Why this works:
      - ProtoSSM sees temporal context across all 12 windows
      - Different starting points expose different context patterns
      - Averaging over 5 views reduces temporal boundary artifacts
    """
    proto_model.eval()
    all_preds = []
    
    emb_t  = torch.tensor(emb_files, dtype=torch.float32)
    sc_t   = torch.tensor(sc_files,  dtype=torch.float32)
    
    for shift in shifts:
        if shift == 0:
            e_shifted = emb_t
            s_shifted = sc_t
        else:
            e_shifted = torch.roll(emb_t, shift, dims=1)
            s_shifted = torch.roll(sc_t,  shift, dims=1)
        
        with torch.no_grad():
            out = proto_model(
                e_shifted, s_shifted,
                site_ids=site_t, hours=hour_t
            ).numpy()   # (n_files, 12, 234)
        
        if shift != 0:
            out = np.roll(out, -shift, axis=1)  # undo shift
        
        all_preds.append(out)
    
    return np.mean(all_preds, axis=0)  # (n_files, 12, 234)

print("✅ CHANGE 3: TTA with 5 circular shifts defined")


In [ ]:
# ── Cell 7j: Residual SSM (second-pass error correction) ──────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualSSM(nn.Module):
    """
    Lightweight second-pass model that learns to correct
    systematic errors from the first-pass ensemble.
    
    Input:  embeddings + first-pass scores (concatenated)
    Output: additive correction to first-pass scores
    
    Key design: output head initialized to zero
    so corrections start small and only grow if helpful.
    ~25s training on 59 files.
    """
    def __init__(self, d_input=1536, d_scores=234,
                 d_model=64, d_state=8,
                 n_classes=234, n_windows=12,
                 dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.n_classes = n_classes

        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))

        self.site_emb  = nn.Embedding(n_sites, meta_dim)
        self.hour_emb  = nn.Embedding(24,      meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc   = nn.Parameter(
            torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_fwd   = SelectiveSSM(d_model, d_state)
        self.ssm_bwd   = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm  = nn.LayerNorm(d_model)
        self.ssm_drop  = nn.Dropout(dropout)

        self.output_head = nn.Linear(d_model, n_classes)
        # Zero init — corrections start at zero, only grow if helpful
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass], dim=-1)
        h = self.input_proj(x) + self.pos_enc[:, :T, :]

        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat(
                [self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings-1)),
                 self.hour_emb(hours.clamp(0, 23))], dim=-1))
            h = h + meta.unsqueeze(1)

        res = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h   = self.ssm_drop(self.ssm_merge(
            torch.cat([h_f, h_b], dim=-1)))
        h   = self.ssm_norm(h + res)

        return self.output_head(h)   # (B, T, n_classes)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters()
                   if p.requires_grad)


def train_residual_ssm(emb_full, first_pass_flat, Y_full,
                       site_ids, hour_ids,
                       n_epochs=30, patience=8, lr=1e-3,
                       correction_weight=0.30,
                       verbose=False):
    """
    Train ResidualSSM to predict (Y - sigmoid(first_pass)).
    Returns corrected flat scores (n_rows, n_classes).
    ~20s on CPU.
    """
    n_files    = len(emb_full) // N_WINDOWS
    emb_f      = emb_full.reshape(n_files, N_WINDOWS, -1)
    fp_f       = first_pass_flat.reshape(n_files, N_WINDOWS, -1)
    lab_f      = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    # Residual target = label - sigmoid(first_pass)
    fp_prob    = 1.0 / (1.0 + np.exp(-np.clip(fp_f, -30, 30)))
    residuals  = lab_f - fp_prob   # values in [-1, 1]

    print(f"Residuals: mean={residuals.mean():.4f}  "
          f"std={residuals.std():.4f}  "
          f"abs_mean={np.abs(residuals).mean():.4f}")

    # Train / val split (file level, no shuffle leakage)
    n_val    = max(1, int(n_files * 0.15))
    rng      = torch.Generator(); rng.manual_seed(42)
    perm     = torch.randperm(n_files, generator=rng).numpy()
    val_i    = perm[:n_val];  train_i = perm[n_val:]

    emb_t    = torch.tensor(emb_f,    dtype=torch.float32)
    fp_t     = torch.tensor(fp_f,     dtype=torch.float32)
    res_t    = torch.tensor(residuals, dtype=torch.float32)
    site_t   = torch.tensor(site_ids, dtype=torch.long)
    hour_t   = torch.tensor(hour_ids, dtype=torch.long)

    model    = ResidualSSM(n_classes=N_CLASSES)
    print(f"ResidualSSM params: {model.count_parameters():,}")

    opt      = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=1e-3)
    sched    = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1,
        pct_start=0.1, anneal_strategy="cos")

    best_loss, best_state, wait = float("inf"), None, 0

    for ep in range(n_epochs):
        model.train()
        corr = model(emb_t[train_i], fp_t[train_i],
                     site_ids=site_t[train_i],
                     hours   =hour_t[train_i])
        loss = F.mse_loss(corr, res_t[train_i])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        model.eval()
        with torch.no_grad():
            val_corr = model(emb_t[val_i], fp_t[val_i],
                             site_ids=site_t[val_i],
                             hours   =hour_t[val_i])
            val_loss = F.mse_loss(val_corr, res_t[val_i])

        if val_loss.item() < best_loss:
            best_loss  = val_loss.item()
            best_state = {k: v.clone()
                          for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            if verbose: print(f"  Early stop ep {ep+1}")
            break

    model.load_state_dict(best_state)
    print(f"ResidualSSM trained — best val MSE={best_loss:.6f}")

    # Apply correction to ALL training data (for verification)
    model.eval()
    with torch.no_grad():
        all_corr = model(emb_t, fp_t,
                         site_ids=site_t,
                         hours   =hour_t).numpy()
    print(f"Correction magnitude: "
          f"mean_abs={np.abs(all_corr).mean():.4f}  "
          f"max={np.abs(all_corr).max():.4f}")

    return model, correction_weight


print("✅ ResidualSSM defined (~439K params, ~20s training)")

In [ ]:
# ── Cell 8: OOF evaluation (train mode only) ──────────────────────────
baseline_auc = None
oof_raw      = None
 
if CFG["run_oof"]:
    print("Running honest OOF evaluation on training data…")
    baseline_auc, oof_raw = honest_oof_auc(
        sc_tr, Y_FULL_aligned, meta_tr,
        n_splits=CFG["oof_n_splits"],
        label="raw Perch"
    )
    print(f"\nBaseline OOF AUC: {baseline_auc:.6f}  ← your starting point")
else:
    print("Submit mode: skipping OOF evaluation")

In [ ]:
# ── Cell 8b: Full Pipeline OOF ─────────────────────────────────────────

def run_pipeline_oof(emb_full, sc_full, Y_full, meta_full, n_splits=5):
    """
    Proper full-pipeline OOF.
    Trains ProtoSSM + MLP on K-1 folds, predicts on held-out fold.
    ~3-4 min total on CPU. Use this instead of the raw-Perch OOF.
    """
    file_meta = (
        meta_full.drop_duplicates("filename")
        .reset_index(drop=True)
    )

    gkf = GroupKFold(n_splits=n_splits)
    oof_probs = np.zeros((len(sc_full), N_CLASSES), dtype=np.float32)

    for fold, (tr_f, va_f) in enumerate(
        gkf.split(file_meta, groups=file_meta["filename"]), 1
    ):
        tr_fnames = set(file_meta.iloc[tr_f]["filename"])
        va_fnames = set(file_meta.iloc[va_f]["filename"])

        tr_mask = meta_full["filename"].isin(tr_fnames).values
        va_mask = meta_full["filename"].isin(va_fnames).values

        emb_tr_f = emb_full[tr_mask]
        sc_tr_f = sc_full[tr_mask]
        Y_tr_f = Y_full[tr_mask]
        meta_tr_f = meta_full[tr_mask].reset_index(drop=True)

        emb_va_f = emb_full[va_mask]
        sc_va_f = sc_full[va_mask]
        meta_va_f = meta_full[va_mask].reset_index(drop=True)

        # ── Train ProtoSSM on train fold ───────────────────────────────
        proto_model, site2i = train_light_proto_ssm(
            emb_tr_f,
            sc_tr_f,
            Y_tr_f,
            meta_tr_f,
            n_epochs=40,
            patience=8,
            lr=1e-3,
            verbose=False,
        )

        # ── ProtoSSM predict on val fold ───────────────────────────────
        n_va = len(emb_va_f) // N_WINDOWS

        va_fn_list = (
            meta_va_f.drop_duplicates("filename")["filename"].tolist()
        )

        va_site_ids = np.array(
            [
                min(
                    site2i.get(
                        meta_va_f.loc[
                            meta_va_f["filename"] == fn, "site"
                        ].iloc[0],
                        0,
                    ),
                    19,
                )
                for fn in va_fn_list
            ],
            dtype=np.int64,
        )

        va_hour_ids = np.array(
            [
                int(
                    meta_va_f.loc[
                        meta_va_f["filename"] == fn, "hour_utc"
                    ].iloc[0]
                )
                % 24
                for fn in va_fn_list
            ],
            dtype=np.int64,
        )

        proto_model.eval()
        with torch.no_grad():
            proto_va = proto_model(
                torch.tensor(
                    emb_va_f.reshape(n_va, N_WINDOWS, -1),
                    dtype=torch.float32,
                ),
                torch.tensor(
                    sc_va_f.reshape(n_va, N_WINDOWS, -1),
                    dtype=torch.float32,
                ),
                site_ids=torch.tensor(va_site_ids, dtype=torch.long),
                hours=torch.tensor(va_hour_ids, dtype=torch.long),
            ).numpy().reshape(-1, N_CLASSES)

        # ── Train MLP on train fold ────────────────────────────────────
        probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
            emb_tr_f,
            sc_tr_f,
            Y_tr_f,
            min_pos=5,
            pca_dim=64,
            alpha_blend=0.4,
        )

        sc_va_mlp = apply_mlp_probes_vectorized(
            emb_va_f,
            sc_va_f,
            probe_models,
            emb_scaler,
            emb_pca,
            alpha_blend,
        )

        # ── Ensemble + sigmoid ─────────────────────────────────────────
        first_pass = 0.5 * proto_va + 0.5 * sc_va_mlp
        probs_va = 1.0 / (1.0 + np.exp(-np.clip(first_pass, -30, 30)))
        oof_probs[va_mask] = probs_va

        fold_auc = macro_auc(Y_full[va_mask], probs_va)
        print(
            f"  Fold {fold}/{n_splits}  val files={len(va_fnames)}  AUC={fold_auc:.6f}"
        )

    overall = macro_auc(Y_full, oof_probs)
    print(f"\nFull pipeline OOF AUC: {overall:.6f}")
    return overall, oof_probs


if CFG["run_oof"]:
    pipeline_auc, oof_pipeline = run_pipeline_oof(
        emb_tr,
        sc_tr,
        Y_FULL_aligned,
        meta_tr,
        n_splits=5,
    )

## Test or Dry-Run Inference

In formal Kaggle reruns, `test_soundscapes` contains the hidden test audio. In normal public notebook runs, that folder is empty, so the notebook falls back to train soundscapes for shape and runtime verification.

For the scored submission, the generated file had **240 rows x 235 columns**, zero NaNs, and produced the current confirmed **0.941 public LB** after Kaggle's competition rerun.


In [ ]:
# ── Cell 9: Test inference ─────────────────────────────────────────────
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
 
if IS_DRY_RUN:
    n = CFG["dryrun_n_files"] or 20
    print(f"No hidden test — dry-run on {n} train files")
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:n]
else:
    print(f"Hidden test files: {len(test_paths)}")
 
meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=CFG["verbose"])
print(f"Test scores: {sc_te.shape}")


## ProtoSSM Branch Output

This cell creates the main sequence-model prediction table. It combines the Perch logits, learned MLP probe signal, priors, calibration, confidence scaling, temporal smoothing, and residual correction into `submission_protossm.csv`.

In the 2026-05-02 run, this branch completed quickly after the Perch cache was built. The training remains small enough for a CPU Kaggle notebook, which is useful because it avoids GPU availability and CUDA compatibility issues.


In [ ]:
# ── Cell 10: Full pipeline with ProtoSSM + ResidualSSM ─────────────────

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

# ── Step A: Train LightProtoSSM ────────────────────────────────────────
t0 = time.time()
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=40, patience=8, lr=1e-3, verbose=False)
print(f"ProtoSSM training: {time.time()-t0:.1f}s")

# ── Step B: Run ProtoSSM on TEST ───────────────────────────────────────
n_test_files  = len(sc_te) // N_WINDOWS
emb_te_f      = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f       = sc_te.reshape(n_test_files, N_WINDOWS, -1)

test_fnames   = meta_te.drop_duplicates("filename")["filename"].tolist()
n_sites_cap   = 20
test_site_ids = np.array([
    min(site2i_tr.get(
        meta_te.loc[meta_te["filename"]==fn,"site"].iloc[0], 0),
        n_sites_cap-1)
    for fn in test_fnames], dtype=np.int64)
test_hour_ids = np.array([
    int(meta_te.loc[meta_te["filename"]==fn,"hour_utc"].iloc[0]) % 24
    for fn in test_fnames], dtype=np.int64)

proto_model.eval()
with torch.no_grad():
    proto_out = proto_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(sc_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()
proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)

# ── Step C: Prior tables ───────────────────────────────────────────────
prior_tables   = build_prior_tables(sc, Y_SC)
sc_te_adjusted = apply_prior(
    sc_te,
    sites=meta_te["site"].to_numpy(),
    hours=meta_te["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.4,
)

# ── Step D: MLP probes ─────────────────────────────────────────────────
probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr, scores_raw=sc_tr, Y=Y_FULL_aligned,
    min_pos=5, pca_dim=64, alpha_blend=0.4,
)
sc_te_adjusted = apply_mlp_probes_vectorized(
    emb_te, sc_te_adjusted,
    probe_models, emb_scaler, emb_pca, alpha_blend,
)

# ── Step E: First-pass ensemble (ProtoSSM + MLP) ───────────────────────
ENSEMBLE_W      = 0.5
first_pass_flat = (ENSEMBLE_W * proto_scores_flat
                   + (1.0 - ENSEMBLE_W) * sc_te_adjusted)

# ── Step F: ResidualSSM (second-pass correction) ───────────────────────
# Build training-data first-pass scores for residual training
n_tr_files    = len(sc_tr) // N_WINDOWS
emb_tr_f      = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f       = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)

tr_fnames     = meta_tr.drop_duplicates("filename")["filename"].tolist()
tr_site_ids   = np.array([
    min(site2i_tr.get(
        meta_tr.loc[meta_tr["filename"]==fn,"site"].iloc[0], 0),
        n_sites_cap-1)
    for fn in tr_fnames], dtype=np.int64)
tr_hour_ids   = np.array([
    int(meta_tr.loc[meta_tr["filename"]==fn,"hour_utc"].iloc[0]) % 24
    for fn in tr_fnames], dtype=np.int64)


# Get ProtoSSM scores on training data
# CORRECT — using emb_tr_f, sc_tr_f, tr_site_ids (train data)
proto_tr_out = run_tta_proto(
    proto_model, emb_tr_f, sc_tr_f,
    site_t=torch.tensor(tr_site_ids, dtype=torch.long),
    hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
    shifts=[0, 1, -1, 2, -2],
)

proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

# Get MLP scores on training data
sc_tr_prior   = apply_prior(
    sc_tr,
    sites=meta_tr["site"].to_numpy(),
    hours=meta_tr["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.4,
)
sc_tr_mlp = apply_mlp_probes_vectorized(
    emb_tr, sc_tr_prior,
    probe_models, emb_scaler, emb_pca, alpha_blend,
)
first_pass_tr = (ENSEMBLE_W * proto_tr_flat
                 + (1.0 - ENSEMBLE_W) * sc_tr_mlp)

train_probs_for_calib = sigmoid(first_pass_tr)
PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
    oof_probs=train_probs_for_calib,
    Y_FULL=Y_FULL_aligned,
    threshold_grid=[0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    n_windows=N_WINDOWS,
)


# Train ResidualSSM on training errors
t0 = time.time()
res_model, correction_weight = train_residual_ssm(
    emb_full=emb_tr,
    first_pass_flat=first_pass_tr,
    Y_full=Y_FULL_aligned,
    site_ids=tr_site_ids,
    hour_ids=tr_hour_ids,
    n_epochs=30,
    patience=8,
    lr=1e-3,
    correction_weight=0.30,
    verbose=False,
)
print(f"ResidualSSM training: {time.time()-t0:.1f}s")

# Apply ResidualSSM correction to TEST scores
first_pass_te_f  = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
res_model.eval()
with torch.no_grad():
    test_correction = res_model(
        torch.tensor(emb_te_f,         dtype=torch.float32),
        torch.tensor(first_pass_te_f,  dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours   =torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()

correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores    = (first_pass_flat
                   + correction_weight * correction_flat)

print(f"Correction applied — "
      f"mean_abs={np.abs(correction_flat).mean():.4f}  "
      f"score range [{final_scores.min():.3f}, {final_scores.max():.3f}]")

# ── Step G: Temperature scaling ────────────────────────────────────────
final_scores = final_scores / temperatures[None, :]

# ── Step H: Sigmoid → probabilities ───────────────────────────────────
probs = sigmoid(final_scores)

# ── Step I: Post-processing pipeline ──────────────────────────────────
probs = file_confidence_scale(probs, n_windows=N_WINDOWS,
                               top_k=2,       power=0.4)
probs = rank_aware_scaling(   probs, n_windows=N_WINDOWS,
                               power=0.4)
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS,
                               base_alpha=0.20)
probs = np.clip(probs, 0.0, 1.0)

# probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)

# ── Step J: Build submission ───────────────────────────────────────────
sub = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
sub.insert(0, "row_id", meta_te["row_id"].values)
assert list(sub.columns) == ["row_id"] + PRIMARY_LABELS
assert len(sub) == len(test_paths) * N_WINDOWS
assert not sub.isna().any().any()
sub.to_csv("submission_protossm.csv", index=False)                                                                                        
protossm_sub = sub.copy()

print(f"\nsubmission.csv saved — shape {sub.shape}")
print(f"Total wall time: {(time.time() - _WALL_START)/60:.1f} min")

del emb_tr_f, sc_tr_f, proto_model, res_model                                                                                             
gc.collect()                                                                                                                              
print("Memory freed. Ready for SED cell.")

## Distilled SED Branch

The second branch loads Tucker Arrants' public distilled SED ONNX folds. This branch looks at mel-spectrogram style evidence and is complementary to Perch embedding sequence modeling.

The important caution is that SED-only dry-run metrics can be optimistic on train soundscape fallback rows. In this solution, the SED branch is used as a moderate blend component rather than as a standalone replacement for the sequence branch.


In [ ]:
# ── Cell 11: Tucker Arrants distilled SED ONNX inference ──────────────

import librosa
from scipy.ndimage import gaussian_filter1d

N_MELS_SED = 256
N_FFT_SED  = 2048
HOP_SED    = 512
FMIN_SED   = 20
FMAX_SED   = 16000
TOP_DB_SED = 80


def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError(
            "sed_fold0.onnx not found. "
            "Attach tuckerarrants/bc2026-distilled-sed-public to this notebook."
        )
    return hits[0].parent


def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    return ort.InferenceSession(
        str(path),
        sess_options=so,
        providers=["CPUExecutionProvider"]
    )


def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0,
        )
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)

    return np.stack(mels)[:, None].astype(np.float32)


def file_to_sed_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)

    if y.ndim == 2:
        y = y.mean(axis=1)

    if sr0 != SR:
        y = librosa.resample(y, orig_sr=sr0, target_sr=SR)

    n = 60 * SR

    if len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    else:
        y = y[:n]

    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC

    return chunks, ends


def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


# Load the 5 SED fold models
sed_dir = find_sed_dir()

sed_fold_paths = sorted(
    sed_dir.glob("sed_fold*.onnx"),
    key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1))
)

sed_sessions = [make_sed_session(p) for p in sed_fold_paths]

print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")


# Run on the exact same test files used by Cell 9/10
sed_rows, sed_preds = [], []
_t0_sed = time.time()


for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_sed_chunks(path)
    mel = audio_to_mel(chunks)

    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)

    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})

        clip_logits = outs[0]             # (12, 234)
        frame_max   = outs[1].max(axis=1) # (12, 234)

        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)

    p_mean = p_sum / len(sed_sessions)

    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(
            p_mean,
            sigma=0.65,
            axis=0,
            mode="nearest"
        ).astype(np.float32)

    stem = path.stem

    sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)

    if i == 1 or i % 50 == 0 or i == len(test_paths):
        print(f"SED: {i}/{len(test_paths)} | {time.time()-_t0_sed:.1f}s")


sed_preds_arr = np.concatenate(sed_preds, axis=0)

sed_sub = pd.DataFrame(
    np.clip(sed_preds_arr, 0.0, 1.0),
    columns=PRIMARY_LABELS
)

sed_sub.insert(0, "row_id", sed_rows)

sed_sub.to_csv("submission_sed.csv", index=False)

print(f"Saved submission_sed.csv: {sed_sub.shape}")

## Final Blend

The final submission uses rank averaging:

- 60% ProtoSSM / Perch sequence branch
- 40% distilled SED branch

Rank blending is more robust than raw probability averaging when component score scales differ. It preserves the ordering signal from both models while reducing calibration mismatch. The notebook writes the final `submission.csv` expected by the competition.

## What did not transfer as well

The surrounding experiments are useful context:

| Candidate family | Public LB / status | Lesson |
| --- | ---: | --- |
| Mattia all-public ProtoSSM+SED | 0.936 | Strong, clean baseline, but below the sequence branch. |
| Ferukk Mattia v2 public clone | 0.937 | Good confirmation that the Mattia family is solid. |
| Youssef exact public V2 | 0.934 | Public page scores may not exactly reproduce across reruns. |
| Tucker/Troy SED-only probes | 0.912 / 0.917 | Train-soundscape dry-run can overestimate SED-only notebooks. |
| Non-addable Enric/Needless/Gerard lines | blocked | High-looking public outputs are not enough if required datasets are private or 403. |

## Takeaways

1. Use all-public, attachable inputs only. A high score is not reproducible if the required model dataset cannot be added to a fresh Kaggle notebook.
2. Keep the output contract boring: exact row order, 234 class columns, no NaNs.
3. Treat local dry-run metrics as a gate, not as a leaderboard simulator.
4. Combine complementary evidence. Perch sequence modeling captures temporal structure; SED folds add spectrogram-local event evidence.
5. Prefer robust blending. Rank averaging worked better than trusting raw probability scales from heterogeneous branches.


In [ ]:
# Cell 3 — V33: Dual gate (PC010 + RANK1) + train-audio HEAD + HGNet downstream

import glob
import numpy as np
import pandas as pd

PROTOSSM_CSV = "submission_protossm.csv"
SED_CSV     = "submission_sed.csv"
OUT_CSV     = "submission_pc010_raw.csv"  # write to RAW so HGNet blend cell still works downstream

EPS = 1e-5
SED_W = 0.40

# calibrated PROTOSSM/ProtoSSM rescue
FAKE_ONLY_THR   = 0.50
SED_LOW_THR     = 0.05
FAKE_ONLY_BLEND = 0.08

# Proto temporal continuity rescue: wider fat-tailed context
PROTO_CONT_RADIUS     = 3
PROTO_CONT_DF         = 2.0
PROTO_CONT_SCALE      = 1.20
PROTO_CONT_RANK_THR   = 0.88
PROTO_LOCAL_RANK_THR  = 0.75
SED_CONT_LOW_THR      = 0.12

# rare local SED spike rescue
SED_ONLY_RANK_THR = 0.95
FAKE_RANK_LOW_THR = 0.80
SED_ONLY_BLEND    = 0.12

# V33 dual-gate
PC010_WEIGHT      = 0.70   # weight for PROTO_CONT_BLEND=0.10 branch
RANK1_WEIGHT      = 1.0 - PC010_WEIGHT
HEAD_RANK_BLEND   = 0.05   # 5% train-audio head into rank1 branch

a = pd.read_csv(PROTOSSM_CSV)
b = pd.read_csv(SED_CSV)

cols = [c for c in a.columns if c != "row_id"]
b = b.set_index("row_id").loc[a["row_id"]].reset_index()

pa = np.clip(a[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
pb = np.clip(b[cols].to_numpy(np.float32), EPS, 1.0 - EPS)

row_ids = a["row_id"].astype(str).to_numpy()
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])

xa = pd.DataFrame(pa).rank(axis=0, pct=True).to_numpy(np.float32)
xb = pd.DataFrame(pb).rank(axis=0, pct=True).to_numpy(np.float32)

# Fat-tailed proto context
offs = np.arange(-PROTO_CONT_RADIUS, PROTO_CONT_RADIUS + 1, dtype=np.float32)
proto_kernel = (1.0 + (offs / PROTO_CONT_SCALE) ** 2 / PROTO_CONT_DF) ** (-(PROTO_CONT_DF + 1.0) / 2.0)
proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)

pa_ctx = pa.copy()
R = PROTO_CONT_RADIUS
for fid in pd.unique(file_ids):
    m = file_ids == fid
    x = pa[m]
    if len(x) > 1:
        xp = np.pad(x, ((R, R), (0, 0)), mode="edge")
        pa_ctx[m] = sum(proto_kernel[i] * xp[i:i + len(x)] for i in range(2 * R + 1))

xctx = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)

fake_only = (pa > FAKE_ONLY_THR) & (pb < SED_LOW_THR)
proto_cont = (
    (xctx > PROTO_CONT_RANK_THR) &
    (xa > PROTO_LOCAL_RANK_THR) &
    (pb < SED_CONT_LOW_THR) &
    (~fake_only)
)
sed_only = (
    (xb > SED_ONLY_RANK_THR) &
    (xa < FAKE_RANK_LOW_THR) &
    (~fake_only) &
    (~proto_cont)
)

def make_gate_pred(proto_cont_blend: float) -> np.ndarray:
    """Build the gate prediction with a configurable proto-continuity blend strength."""
    p = xa * (1.0 - SED_W) + xb * SED_W
    p = np.where(fake_only, (1.0 - FAKE_ONLY_BLEND) * p + FAKE_ONLY_BLEND * xa, p)
    p = np.where(
        proto_cont,
        (1.0 - proto_cont_blend) * p + proto_cont_blend * np.maximum(xa, xctx),
        p,
    )
    p = np.where(sed_only, (1.0 - SED_ONLY_BLEND) * p + SED_ONLY_BLEND * xb, p)
    return p.astype(np.float32)

pred_pc010 = make_gate_pred(0.10)  # original PC010
pred_rank1 = make_gate_pred(0.15)  # stronger continuity blend

# Train-audio linear head: 1.15 MB precomputed Perch-embedding logistic regression
# trained on full 35,549 train_audio files. Free signal — one matmul against emb_te.
HEAD_AVAILABLE = False
head_candidates = sorted(glob.glob("/kaggle/input/**/head_weights_train_audio.npz", recursive=True))
if head_candidates:
    try:
        head_weights = np.load(head_candidates[0], allow_pickle=True)
        head_W = head_weights["W"].astype(np.float32)
        head_b = head_weights["b"].astype(np.float32)
        head_mask = head_weights["trained_mask"].astype(bool).reshape(1, -1)
        if emb_te.shape[1] == head_W.shape[1]:
            head_logits = emb_te.astype(np.float32) @ head_W.T + head_b
            head_logits = head_logits * head_mask.astype(np.float32)
            head_probs = 1.0 / (1.0 + np.exp(-np.clip(head_logits, -30, 30)))
            xc = pd.DataFrame(np.clip(head_probs, EPS, 1.0 - EPS)).rank(axis=0, pct=True).to_numpy(np.float32)
            pred_rank1 = (1.0 - HEAD_RANK_BLEND) * pred_rank1 + HEAD_RANK_BLEND * xc
            HEAD_AVAILABLE = True
            print(f"Train-audio HEAD applied @ {HEAD_RANK_BLEND:.3f} rank blend on RANK1 branch ({head_mask.sum()}/{head_mask.size} classes covered)")
        else:
            print(f"HEAD dim mismatch: emb_te={emb_te.shape[1]} W={head_W.shape[1]} — skipping")
    except Exception as e:
        print(f"HEAD load failed: {e} — skipping")
else:
    print("Train-audio HEAD weights not found — skipping")

# Dual gate ensemble
pred = PC010_WEIGHT * pred_pc010 + RANK1_WEIGHT * pred_rank1
print(f"Dual gate: PC010={PC010_WEIGHT:.2f} + RANK1(blend=0.15{', +HEAD' if HEAD_AVAILABLE else ''})={RANK1_WEIGHT:.2f}")

sub = a.copy()
sub[cols] = pred.astype(np.float32)
sub.to_csv(OUT_CSV, index=False)
print(f"Saved {OUT_CSV}: {sub.shape}")

if IS_DRY_RUN:
    print("Dry-run row_ids do not match public sample; writing sample-aligned validation submission.")
    sample_public = pd.read_csv(BASE / "sample_submission.csv")
    template = sub[cols].mean(axis=0).astype(np.float32)
    aligned = sample_public.copy()
    for label in cols:
        aligned[label] = template[label]
    aligned.to_csv("submission.csv", index=False)
else:
    sub.to_csv("submission.csv", index=False)

In [ ]:
# Robust offline install for HGNet/OpenVINO wheels.
# Kaggle may mount notebook inputs under different /kaggle/input names, so do not
# hard-code /kaggle/input/notebooks/<author>/<slug>.
import subprocess
import sys
from pathlib import Path

req_candidates = []
for req in Path("/kaggle/input").rglob("requirements.txt"):
    wheel_dir = req.parent / "wheels"
    if wheel_dir.exists():
        txt = req.read_text(errors="ignore").lower()
        if "openvino" in txt and "onnxruntime" in txt:
            req_candidates.append(req)

if not req_candidates:
    shown = [str(p) for p in Path("/kaggle/input").rglob("requirements.txt")][:20]
    raise FileNotFoundError(
        "Could not find offline HGNet wheel requirements under /kaggle/input. "
        f"Available requirements samples: {shown}"
    )

REQ = sorted(req_candidates, key=lambda p: len(str(p)))[0]
WHEEL_DIR = REQ.parent / "wheels"
print(f"Installing HGNet wheels from {REQ}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "-r",
        str(REQ),
        "--find-links",
        str(WHEEL_DIR),
    ],
    check=True,
)
print("HGNet wheels installed")


In [ ]:
RANDOM_SEED = 1086

import os
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

In [ ]:
import gc
import copy
import random
import typing as tp
from pathlib import Path

from time import time
import numpy as np
import pandas as pd

from tqdm.notebook import tqdm, trange

import soundfile

import timm
import torch
import torchaudio
from torchvision.transforms import v2 as tvt_v2

from torch import nn

from joblib import load as jb_load, Parallel, delayed
import threading

import openvino as ov

In [ ]:
ROOT = Path.cwd().parent

INPUT = ROOT / "input"
DATA = INPUT / "competitions" / "birdclef-2026"
TRAIN_AUDIO = DATA / "train_audio"
TRAIN_SS = DATA / "train_soundscapes"
TEST_SS = DATA / "test_soundscapes"

# Pin to 256x256 model variant — matches our 5-sec / lms_shape=(256,256) pipeline.
# The 256x512 variant expects 10-sec windows and a (1,256,512) input port.
HGNET_SUFFIX = "_256x256"
model_hits = sorted(Path("/kaggle/input").rglob(f"best_model_fold0{HGNET_SUFFIX}.xml"))
if not model_hits:
    # Fall back to unsuffixed (only safe if it is also 256x256)
    model_hits = sorted(Path("/kaggle/input").rglob("best_model_fold0.xml"))
    if model_hits:
        HGNET_SUFFIX = ""
if not model_hits:
    shown = [str(p) for p in Path("/kaggle/input").rglob("best_model_fold0*.xml")][:20]
    raise FileNotFoundError(
        "Could not find HGNet OpenVINO 256x256 model under /kaggle/input. "
        f"Available samples: {shown}"
    )
TRAINED_MODEL = model_hits[0].parent
print(f"HGNet model dir: {TRAINED_MODEL}")
print(f"HGNet filename suffix: {HGNET_SUFFIX!r}")

N_FOLDS = 4
N_CLASSES = 234

DEBUG = True
RANK_AVG = False


In [ ]:
taxonomy = pd.read_csv(DATA / "taxonomy.csv")

CLASSES = taxonomy.primary_label.values.tolist()

label2idx = {label: idx for idx, label in enumerate(taxonomy.primary_label.values)}
idx2label = {idx: label for label, idx in label2idx.items()}

In [ ]:
def set_random_seed(seed: int = 42, deterministic: bool = True):
    """Set seeds"""
    os.environ["PYTHONHASHSEED"] = str(seed)  # python
    random.seed(seed)  # python
    np.random.seed(seed)  # cpu
    torch.manual_seed(seed)  # cpu
    if torch.cuda.is_available():  # gpu
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic

set_random_seed(RANDOM_SEED)

In [ ]:
test_paths = sorted((DATA / "test_soundscapes").glob("*.ogg"))
if len(test_paths) > 0:
    test_ss_paths = test_paths
    print(f"Hidden test files: {len(test_ss_paths)}")
else:
    # Dry-run during save: MUST use the same files as the Perch pipeline (first 20 sorted
    # train_soundscapes .ogg files), otherwise the two outputs won't share row_ids and the
    # ensemble will fail. Aligned with nb1's fallback: sorted(...glob("*.ogg"))[:20].
    test_ss_paths = sorted(TRAIN_SS.glob("*.ogg"))[:20]
    print(f"No hidden test — dry-run on {len(test_ss_paths)} train_soundscapes files (aligned with Perch fallback)")

sample_sub = pd.read_csv(DATA / "sample_submission.csv")
# IS_TEST_ENV = len(sample_sub) > 10

# if IS_TEST_ENV:
#     # about 600 audio files in test environment
#     test_ss_paths = []
#     added = set()
#     for row_id in sample_sub["row_id"].values:
#         file_id = "_".join(row_id.split("_")[:-1])
#         if file_id in added:
#             continue
#         added.add(file_id)
#         test_ss_paths.append(TEST_SS / f"{file_id}.ogg")
# else:
#     if DEBUG:
#         # debug by 600 audio files in train_soundscapes 
#         test_ss_paths = sorted(TRAIN_SS.iterdir())[:600]
#     else:
#         # fast submit
#         test_ss_paths = sorted(TRAIN_SS.iterdir())[:10]

In [ ]:
test_ss_segs = []
for p in test_ss_paths:
    for i in range(0, 60, 5):
        test_ss_segs.append(f"{p.stem}_{i + 5}")

In [ ]:
for row_id in test_ss_segs[:24]:
    print(row_id)

In [ ]:
class LogMelSpectrogramTransform(nn.Module):
    """"""
    def __init__(self, mel_spectrogram_params: tp.Dict, top_db: float, lms_shape=tp.Tuple[int, int]):
        """"""
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(**mel_spectrogram_params)
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=top_db)
        self.resize = tvt_v2.Resize(size=lms_shape)

    @torch.no_grad()
    def forward(self, wave):
        """
        wave: (B, sampling_rate * segment_sec)
        """
        mel_spec = self.mel_transform(wave)
        lms = self.db(mel_spec)  # shape: (B, n_mels, time)
        lms = self.resize(lms)   # shape: (B, *(lms_shape))
        
        batch_size = lms.shape[0]
        lms_flat = lms.reshape(batch_size, -1)
        lms_min = lms_flat.min(dim=1)[0][:, None, None]
        lms_max = lms_flat.max(dim=1)[0][:, None, None]
        lms = (lms - lms_min) / (lms_max - lms_min + 1e-7)

        return lms[:, None, :, :]

In [ ]:
mel_spectrogram_params = dict(
    sample_rate= 32_000,
    n_fft      = 2048,
    win_length = 626,
    hop_length = 313,
    f_min      = 20,
    n_mels     = 256,
    power      = 2.0,
    center     = True,
    pad_mode   = "reflect",
    norm       = "slaney",
    mel_scale  = 'htk',
)
top_db = 80
lms_shape = (256, 256)

lms_transform = LogMelSpectrogramTransform(
    mel_spectrogram_params, top_db=top_db, lms_shape=lms_shape).eval()

In [ ]:
def compile_ov_model(ov_model_path):
    compiled_model = ov.compile_model(
        str(ov_model_path), "CPU", {
            "PERFORMANCE_HINT": "THROUGHPUT",
            "INFERENCE_NUM_THREADS": 4,
            'NUM_STREAMS': 2}
    )
    return compiled_model

In [ ]:
ov_model_list = []
for fold_id in range(N_FOLDS):
    compiled_model = compile_ov_model(
        str(TRAINED_MODEL / f"best_model_fold{fold_id}{HGNET_SUFFIX}.xml"))
    ov_model_list.append(compiled_model)
    del compiled_model

In [ ]:
wave_list = []
shifted_wave_list = []
sample_rate = 32_000 
max_sec = 60
duration = sample_rate * max_sec
shift = int(sample_rate * 2.5)

for path in tqdm(test_ss_paths):

    with soundfile.SoundFile(path) as f:
        n_frames = f.frames

        wave = np.zeros(shift + duration + shift, dtype="float32")

        if n_frames < duration + shift:
            wave[shift:shift + n_frames] = f.read(dtype="float32")
        else:
            wave[shift:shift + duration + shift] = f.read(frames=duration + shift, dtype="float32")
    
    for seg_sec in range(0, max_sec, 5):
        wave_list.append(
            wave[shift + seg_sec * sample_rate: shift + (seg_sec + 5) * sample_rate])
    for seg_sec in range(0, max_sec + 5, 5):
        shifted_wave_list.append(
            wave[seg_sec * sample_rate: (seg_sec + 5) * sample_rate])   



num_test_ss_audios = len(test_ss_paths)  
num_test_ss_segs = len(wave_list)
num_shifted_test_ss_segs = len(shifted_wave_list)

print(num_test_ss_segs, num_test_ss_segs / 12)
print(num_shifted_test_ss_segs, num_shifted_test_ss_segs / 13)

In [ ]:
wave_batches = []
shifted_wave_batches = []
batch_size = 12

for i in trange(0, len(wave_list), batch_size):
    wave_batches.append(np.stack(wave_list[i: i + batch_size], axis=0))

for i in trange(0, len(shifted_wave_list), batch_size):
    shifted_wave_batches.append(np.stack(shifted_wave_list[i: i + batch_size], axis=0))

In [ ]:
lms_batches = Parallel(n_jobs=4, verbose=1)(
    delayed(lms_transform)(torch.from_numpy(waves)) for waves in wave_batches
)
shifted_lms_batches = Parallel(n_jobs=4, verbose=1)(
    delayed(lms_transform)(torch.from_numpy(waves)) for waves in shifted_wave_batches
)

In [ ]:
lms_batches = [
    np.ascontiguousarray(lms, dtype=np.float32) for lms in lms_batches]
shifted_lms_batches = [
    np.ascontiguousarray(lms, dtype=np.float32) for lms in shifted_lms_batches]

del wave_batches, shifted_wave_batches
gc.collect()

In [ ]:
def async_infer_with_order(model, lms_batches, num_requests=4):
    """"""
    idx_batches = []
    tmp_idx = 0
    for b in lms_batches:
        b_size = len(b)
        idx_batches.append(np.arange(tmp_idx, tmp_idx + b_size))
        tmp_idx += b_size
    n_records = tmp_idx

    infer_queue = ov.AsyncInferQueue(model, num_requests)
    
    # array for predict result
    logit_arr = np.zeros((n_records, N_CLASSES), dtype=np.float32)
    
    start_time = time()
    
    def callback(request, userdata):
        input_idxs = userdata
        output = request.get_output_tensor().data
        logit_arr[input_idxs] = output

    infer_queue.set_callback(callback)
    input_name = model.inputs[0].get_any_name()
    
    for idxs, lms in zip(idx_batches, lms_batches):
        infer_queue.start_async({input_name: lms}, userdata=idxs)

    infer_queue.wait_all()

    print(f"... Done by {time() - start_time:.2f} sec")
    return logit_arr

In [ ]:
def rank_normalize(x):
    r_x = np.zeros_like(x) 
    for i in range(x.shape[1]):
        r_x_i = pd.Series(x[:, i]).rank(method="max")
        r_x[:, i] = r_x_i / r_x_i.shape[0]
    return r_x

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
test_preds_arr = np.zeros(
    (len(ov_model_list), num_test_ss_segs, N_CLASSES), dtype=np.float32)

for model_id, ov_model in enumerate(ov_model_list):
    print(f"[model: {model_id}]")
    test_pred = async_infer_with_order(ov_model, lms_batches, 16)
    test_preds_arr[model_id] = test_pred

In [ ]:
shifted_test_preds_arr = np.zeros(
    (len(ov_model_list), num_shifted_test_ss_segs, N_CLASSES), dtype=np.float32)

for model_id, ov_model in enumerate(ov_model_list):
    print(f"[model: {model_id}]")
    test_pred = async_infer_with_order(ov_model, shifted_lms_batches, 16)
    shifted_test_preds_arr[model_id] = test_pred

In [ ]:
if RANK_AVG:
    for model_id in range(len(ov_model_list)):
        test_preds_arr[model_id] = rank_normalize(test_preds_arr[model_id])
    for model_id in range(len(ov_model_list)):
        shifted_test_preds_arr[model_id] = rank_normalize(shifted_test_preds_arr[model_id])
else:
    for model_id in range(len(ov_model_list)):
        test_preds_arr[model_id] = sigmoid(test_preds_arr[model_id])
    for model_id in range(len(ov_model_list)):
        shifted_test_preds_arr[model_id] = sigmoid(shifted_test_preds_arr[model_id])

In [ ]:
print(test_preds_arr.shape, shifted_test_preds_arr.shape)
# # reshape
test_preds_arr_by_record = test_preds_arr.reshape(N_FOLDS, num_test_ss_audios, 12, N_CLASSES)
shifted_test_preds_arr_by_record = shifted_test_preds_arr.reshape(
    N_FOLDS, num_test_ss_audios, 13, N_CLASSES)

print(test_preds_arr_by_record.shape, shifted_test_preds_arr_by_record.shape)

# # weighted avg across shifted preds
test_preds_arr_tta_by_record = (
    0.25 * shifted_test_preds_arr_by_record[..., 0:12, :] +  # 2.5 sec before
    0.50 * test_preds_arr_by_record +
    0.25 * shifted_test_preds_arr_by_record[..., 1:13, :]    # 2.5 sec after
)
# test_preds_arr_tta_by_record = test_preds_arr_by_record
print(test_preds_arr_tta_by_record.shape)

test_preds_arr_tta = test_preds_arr_tta_by_record.reshape(N_FOLDS, num_test_ss_segs, N_CLASSES)
print(test_preds_arr_tta.shape)

# # avg accross folds
test_pred_arr_tta = test_preds_arr_tta.mean(axis=0)
print(test_pred_arr_tta.shape)

In [ ]:
sub_df = pd.DataFrame(
    test_pred_arr_tta,
    columns=CLASSES, 
    index=pd.Series(test_ss_segs, name="row_id"),
).reset_index()

display(sub_df.head())

In [ ]:
# sub_df = pd.merge(
#     sample_sub[["row_id"]], sub_df,
#     on="row_id", how="left")

# ── merged: stash HGNet probabilities for later ensemble ──────────────
probs_hgnet_df = sub_df.copy()
print(f"✅ HGNet pipeline done — probs_hgnet_df shape={probs_hgnet_df.shape}")
print(f"   hgnet value range: [{probs_hgnet_df.iloc[:,1:].values.min():.6f}, "
      f"{probs_hgnet_df.iloc[:,1:].values.max():.6f}]")

In [ ]:
# Final: conservative pc010 + HGNet rank blend

import numpy as np
import pandas as pd

PC010_RAW_CSV = "submission_pc010_raw.csv"
OUT_CSV = "submission.csv"

PC010_WEIGHT = 0.920000
HGNET_WEIGHT = 1.0 - PC010_WEIGHT

pc = pd.read_csv(PC010_RAW_CSV)
hg = probs_hgnet_df.copy()

cols = [c for c in pc.columns if c != "row_id"]
assert set(cols) == set([c for c in hg.columns if c != "row_id"]), "HGNet label columns mismatch"

hg = hg.set_index("row_id").loc[pc["row_id"]].reset_index()
hg.to_csv("submission_hgnet_raw.csv", index=False)
print(f"Saved submission_hgnet_raw.csv before sample alignment: {hg.shape}")

def rank_normalize_cols(x: np.ndarray) -> np.ndarray:
    out = np.empty_like(x, dtype=np.float32)
    for j in range(x.shape[1]):
        out[:, j] = pd.Series(x[:, j]).rank(method="average", pct=True).to_numpy(np.float32)
    return out

pc_arr = np.clip(pc[cols].to_numpy(np.float32), 1e-6, 1.0 - 1e-6)
hg_arr = np.clip(hg[cols].to_numpy(np.float32), 1e-6, 1.0 - 1e-6)

pc_rank = rank_normalize_cols(pc_arr)
hg_rank = rank_normalize_cols(hg_arr)

pred = PC010_WEIGHT * pc_rank + HGNET_WEIGHT * hg_rank
print(
    f"Applied conservative pc010+HGNet rank blend: "
    f"pc010={PC010_WEIGHT:.3f} hgnet={HGNET_WEIGHT:.3f}"
)
print(
    f"HGNet delta vs pc010: corr={np.corrcoef(pc_rank.ravel(), hg_rank.ravel())[0, 1]:.6f} "
    f"mae={np.mean(np.abs(pc_rank - hg_rank)):.6f}"
)

sub = pc.copy()
sub[cols] = pred.astype(np.float32)
sub.to_csv("submission_blend_raw.csv", index=False)
print(f"Saved submission_blend_raw.csv before sample alignment: {sub.shape}")

if IS_DRY_RUN:
    print("Dry-run row_ids do not match public sample; writing sample-aligned validation submission.")
    sample_public = pd.read_csv(BASE / "sample_submission.csv")
    template = sub[cols].mean(axis=0).astype(np.float32)
    sub = sample_public.copy()
    for label in cols:
        sub[label] = template[label]

assert list(sub.columns) == ["row_id"] + cols
assert not sub.isna().any().any()
vals = sub[cols].to_numpy(np.float32)
assert np.isfinite(vals).all()
assert vals.min() >= 0.0 and vals.max() <= 1.0
sub.to_csv(OUT_CSV, index=False)
print(f"submission.csv written after pc010+HGNet blend: {sub.shape}")


## Hyperbolic Taxonomic Prototypes

Each species k is represented by a prototype p_k on the Poincare ball B^32
of constant negative curvature. Prototypes are initialised by recursive
tree embedding (Sarkar, 2011) of the kingdom -> class -> order -> family ->
genus -> species hierarchy, then refined jointly with a Euclidean-to-ball
projector phi: R^1536 -> B^32. Per-class scores are -alpha_k * d_B(phi(x), p_k)
+ beta_k, where d_B is the Poincare distance.

Hyperbolic geometry matches the multiplicative branching of biological
taxonomy: tree distances embed with distortion 1 + epsilon (Sarkar, 2011),
whereas Euclidean embeddings of N-leaf trees incur distortion Omega(sqrt(N)).
The construction lets rare species share representational mass with their
genus and family, which is well-suited to the padded class-mean AP metric.

The branch consists of four cells: Sarkar initialisation, head definition,
training on the cached Perch embeddings, and inference on the test windows.
Its output is a per-window probability matrix written to submission_htp.csv
and consumed by the final rank blend.


In [ ]:
# Sarkar tree embedding of the BirdCLEF taxonomy.
#
# Produces a prototype matrix P in R^{N_CLASSES x HTP_DIM} whose rows lie
# strictly inside the Poincare ball and whose pairwise hyperbolic distances
# approximate the taxonomic tree distances. Prototypes are initialisation
# only; they become trainable parameters in the next cell.

import math

import numpy as np
import pandas as pd

HTP_DIM   = 48     # Poincare ball dimension
HTP_TAU   = 0.40   # logit temperature applied at scoring time
HTP_SCALE = 0.85   # parent-to-child Riemannian step (decays with depth)
HTP_BLEND = 0.04   # weight contributed to the final rank blend
HTP_OK    = False  # gate flag toggled by the training cell

try:
    _tax = taxonomy.copy()
except NameError:
    _tax = pd.read_csv(BASE / "taxonomy.csv")


def _row_path(row):
    fields = []
    for col in ["class_name", "order", "family", "genus", "primary_label"]:
        v = row.get(col, "")
        fields.append(f"{col}:{v}" if isinstance(v, str) and v else f"{col}:UNK")
    return ["ROOT"] + fields


_paths = [_row_path(r) for _, r in _tax.iterrows()]

_children: dict[str, list[str]] = {}
for p in _paths:
    for u, v in zip(p, p[1:]):
        _children.setdefault(u, [])
        if v not in _children[u]:
            _children[u].append(v)


def _exp0_ball(direction: np.ndarray, scale: float) -> np.ndarray:
    """Exponential map at the origin of the Poincare ball."""
    n = np.linalg.norm(direction)
    if n < 1e-9:
        return np.zeros_like(direction)
    return (direction / n) * math.tanh(scale)


def _mobius_add(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Mobius addition x (+) y on the Poincare ball with curvature -1."""
    xy = float(np.dot(x, y))
    xx = float(np.dot(x, x))
    yy = float(np.dot(y, y))
    num = (1 + 2 * xy + yy) * x + (1 - xx) * y
    den = 1 + 2 * xy + xx * yy + 1e-9
    return num / den


def _unit_directions(n: int, dim: int, seed: int) -> np.ndarray:
    """Deterministic near-uniform unit vectors on S^{dim-1}."""
    rng = np.random.RandomState(seed)
    g = rng.normal(size=(n, dim))
    g /= np.maximum(np.linalg.norm(g, axis=1, keepdims=True), 1e-9)
    return g


proto_node: dict[str, np.ndarray] = {"ROOT": np.zeros(HTP_DIM, dtype=np.float64)}
_seed_iter = 1234


def _place(node: str, depth: int):
    """Place each child by parent (+) exp_0(step * direction)."""
    global _seed_iter
    kids = _children.get(node, [])
    if not kids:
        return
    dirs = _unit_directions(len(kids), HTP_DIM, seed=_seed_iter)
    _seed_iter += 1
    parent_pos = proto_node[node]
    step = HTP_SCALE / max(depth, 1)
    for k, d in zip(kids, dirs):
        proto_node[k] = _mobius_add(parent_pos, _exp0_ball(d, step))
        _place(k, depth + 1)


_place("ROOT", 1)

_species_keys = [f"primary_label:{lab}" for lab in PRIMARY_LABELS]
_missing = [s for s in _species_keys if s not in proto_node]
if _missing:
    print(f"[HTP] {len(_missing)} species missing taxonomic ancestry; placed at origin")

_P = np.stack([
    proto_node.get(s, np.zeros(HTP_DIM, dtype=np.float64)) for s in _species_keys
]).astype(np.float32)

# Project any boundary points back into the open unit ball.
_norms = np.linalg.norm(_P, axis=1, keepdims=True)
_P = np.where(_norms > 0.95, _P / _norms * 0.95, _P).astype(np.float32)

HTP_PROTOTYPES = _P
print(f"[HTP] prototypes  shape={_P.shape}  max_norm={np.linalg.norm(_P, axis=1).max():.4f}")


In [ ]:
# Hyperbolic prototype head.
#
# Forward pass:  x in R^{1536}
#   z   = tanh(||Wx+b||) * (Wx+b) / ||Wx+b||  * 0.97        (projection into B^d)
#   d_k = arccosh(1 + 2 * ||z - p_k||^2 / ((1-||z||^2)(1-||p_k||^2)))
#   l_k = -alpha_k * d_k + beta_k
#
# alpha_k > 0 (per-class scale on distance) and beta_k (per-class offset) are
# learnable. The 0.97 factor keeps z strictly inside the open ball, which is
# needed for arccosh to be finite.

import torch
import torch.nn as nn
import torch.nn.functional as F


def _poincare_dist(u: torch.Tensor, v: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """Pairwise Poincare distance between rows of u (B, d) and v (K, d)."""
    u2 = (u * u).sum(dim=-1, keepdim=True).clamp(max=1.0 - eps)
    v2 = (v * v).sum(dim=-1).clamp(max=1.0 - eps)
    diff = u.unsqueeze(1) - v.unsqueeze(0)
    dnum = (diff * diff).sum(dim=-1)
    denom = (1.0 - u2) * (1.0 - v2.unsqueeze(0)) + eps
    return torch.acosh((1.0 + 2.0 * dnum / denom).clamp(min=1.0 + eps))


class HTPHead(nn.Module):
    def __init__(self, in_dim: int, ball_dim: int, prototypes: np.ndarray, tau: float):
        super().__init__()
        self.proj = nn.Linear(in_dim, ball_dim, bias=True)
        nn.init.xavier_uniform_(self.proj.weight, gain=0.1)
        nn.init.zeros_(self.proj.bias)
        self.protos    = nn.Parameter(torch.tensor(prototypes, dtype=torch.float32))
        self.log_alpha = nn.Parameter(torch.zeros(prototypes.shape[0]))
        self.log_beta  = nn.Parameter(torch.zeros(prototypes.shape[0]))
        self.tau = tau

    def _squash(self, x: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
        n = x.norm(dim=-1, keepdim=True).clamp(min=eps)
        return torch.tanh(n) / n * x * 0.97

    def _constrain(self) -> torch.Tensor:
        n = self.protos.norm(dim=-1, keepdim=True).clamp(min=1e-7)
        return self.protos * torch.where(n > 0.95, 0.95 / n, torch.ones_like(n))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self._squash(self.proj(x))
        d = _poincare_dist(z, self._constrain())
        alpha = F.softplus(self.log_alpha) + 0.1
        return -alpha.unsqueeze(0) * d + self.log_beta.unsqueeze(0)


In [ ]:
# Train HTPHead on cached Perch embeddings.
#
# Loss:        BCE-with-logits, class-balanced via pos_weight = (N_neg / N_pos).clip(1, 50)
# Optimiser:   AdamW (lr=3e-3, wd=1e-5)
# Regulariser: input dropout 0.1; gradient norm clipped at 5.0
# Schedule:    30 epochs, batch size 4096, cosine LR decay to 1e-4
# Device:      CPU (Kaggle inference environment)

import math
import time

HTP_EPOCHS  = 30
HTP_BS      = 4096
HTP_LR_MAX  = 3e-3
HTP_LR_MIN  = 1e-4
HTP_WD      = 1e-5
HTP_DROPOUT = 0.1
HTP_DEVICE  = "cpu"

try:
    _t0 = time.time()

    assert emb_tr.ndim == 2 and emb_tr.shape[1] == 1536
    assert Y_FULL_aligned.shape == (emb_tr.shape[0], N_CLASSES)
    assert HTP_PROTOTYPES.shape == (N_CLASSES, HTP_DIM)

    htp_model = HTPHead(emb_tr.shape[1], HTP_DIM, HTP_PROTOTYPES, HTP_TAU).to(HTP_DEVICE)
    htp_opt   = torch.optim.AdamW(htp_model.parameters(), lr=HTP_LR_MAX, weight_decay=HTP_WD)

    pos = Y_FULL_aligned.sum(axis=0).clip(min=1.0)
    neg = Y_FULL_aligned.shape[0] - pos
    pos_weight = torch.tensor(
        (neg / pos).clip(min=1.0, max=50.0), dtype=torch.float32, device=HTP_DEVICE,
    )

    X_t = torch.tensor(emb_tr, dtype=torch.float32)
    Y_t = torch.tensor(Y_FULL_aligned, dtype=torch.float32)
    n = X_t.shape[0]
    rng = np.random.RandomState(0)

    for epoch in range(HTP_EPOCHS):
        # Cosine LR decay from HTP_LR_MAX -> HTP_LR_MIN over the full schedule.
        cos_t = epoch / max(HTP_EPOCHS - 1, 1)
        lr_now = HTP_LR_MIN + 0.5 * (HTP_LR_MAX - HTP_LR_MIN) * (1.0 + math.cos(math.pi * cos_t))
        for g in htp_opt.param_groups:
            g["lr"] = lr_now

        perm = rng.permutation(n)
        running, n_batches = 0.0, 0
        for i in range(0, n, HTP_BS):
            idx = perm[i:i + HTP_BS]
            xb = X_t[idx].to(HTP_DEVICE)
            yb = Y_t[idx].to(HTP_DEVICE)
            if HTP_DROPOUT > 0:
                xb = xb * (torch.rand_like(xb) > HTP_DROPOUT).float()
            loss = F.binary_cross_entropy_with_logits(
                htp_model(xb), yb, pos_weight=pos_weight, reduction="mean",
            )
            htp_opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(htp_model.parameters(), 5.0)
            htp_opt.step()
            running += float(loss.item())
            n_batches += 1
        if epoch == 0 or (epoch + 1) % 5 == 0 or epoch == HTP_EPOCHS - 1:
            print(f"[HTP] epoch {epoch+1}/{HTP_EPOCHS}  lr={lr_now:.5f}  loss={running/max(n_batches,1):.4f}")

    htp_model.eval()
    print(f"[HTP] training complete in {time.time() - _t0:.1f}s")
    HTP_OK = True

except Exception as exc:
    print(f"[HTP] training disabled: {exc}")
    HTP_OK = False


In [ ]:
# Apply HTPHead to the test-window Perch embeddings and write submission_htp.csv.

import numpy as np
import pandas as pd

if HTP_OK:
    try:
        assert emb_te.ndim == 2 and emb_te.shape[1] == 1536

        with torch.no_grad():
            xb = torch.tensor(emb_te, dtype=torch.float32)
            chunks = [htp_model(xb[i:i + 8192]).cpu().numpy()
                      for i in range(0, xb.shape[0], 8192)]
            htp_logits = np.concatenate(chunks, axis=0).astype(np.float32)

        htp_probs = 1.0 / (1.0 + np.exp(-np.clip(htp_logits, -30.0, 30.0)))

        if "row_id" in meta_te.columns:
            htp_rows = meta_te["row_id"].astype(str).tolist()
        else:
            stems = meta_te["filename"].str.replace(".ogg", "", regex=False).to_numpy()
            ends = (
                meta_te["end_sec"].astype(int).to_numpy()
                if "end_sec" in meta_te.columns
                else (meta_te["window_idx"].astype(int).to_numpy() + 1) * 5
            )
            htp_rows = [f"{s}_{e}" for s, e in zip(stems, ends)]

        if len(htp_rows) != htp_probs.shape[0]:
            raise RuntimeError(
                f"row count mismatch: meta_te {len(htp_rows)} vs HTP probs {htp_probs.shape[0]}"
            )

        htp_sub = pd.DataFrame(np.clip(htp_probs, 0.0, 1.0), columns=PRIMARY_LABELS)
        htp_sub.insert(0, "row_id", htp_rows)
        htp_sub.to_csv("submission_htp.csv", index=False)
        print(f"[HTP] submission_htp.csv  shape={htp_sub.shape}  "
              f"range=[{htp_sub.iloc[:,1:].values.min():.6f}, "
              f"{htp_sub.iloc[:,1:].values.max():.6f}]")
    except Exception as exc:
        print(f"[HTP] inference disabled: {exc}")
        HTP_OK = False
else:
    print("[HTP] inference skipped")


## Complete OOF on labeled soundscapes (report-only)

Runs HGNet + distilled SED on the labeled soundscape files,
K-folds HTP, and combines them into an approximation of the
final blend scored with `macro_auc`. Gated on `IS_DRY_RUN`,
try/except-wrapped; never affects submission.csv.


In [ ]:
# Complete OOF, part 1: HGNet on the labeled soundscape files.
# Same pipeline as the test-side HGNet pass (5 OV folds, lms_transform,
# async_infer_with_order, +-2.5s shifted view weighted 0.25/0.50/0.25),
# applied to the ~59 files in `sc`. Output: hgnet_oof_df aligned to meta_tr
# row_ids. Gated on IS_DRY_RUN so the ~15-25 min cost only hits public
# commits, not the formal rerun.

import time as _time_mod
# V33 cell 42 binds `time` to the function (`from time import time`) and
# V33's async_infer_with_order relies on that. An intervening cell (HTP
# training) shadows `time` back to the module, breaking async_infer; rebind.
from time import time as time  # noqa: F401  -- intentional re-rebind

import numpy as np
import pandas as pd

HGNET_OOF_DF = None  # (meta_tr-aligned) HGNet probs on the labeled soundscapes

if globals().get("IS_DRY_RUN", False):
    try:
        _t0 = _time_mod.time()
        import soundfile
        from joblib import Parallel, delayed

        # The labeled soundscape file list.
        _oof_fnames = sorted(sc["filename"].unique().tolist())
        _oof_paths  = [TRAIN_SS / fn for fn in _oof_fnames if (TRAIN_SS / fn).exists()]
        if len(_oof_paths) == 0:
            raise RuntimeError("no labeled soundscape files found under TRAIN_SS")
        _n_oof_files = len(_oof_paths)
        print(f"[OOF/HGNet] running on {_n_oof_files} labeled soundscape files...")

        # Build wave_list + shifted_wave_list (same recipe as the test path).
        _wave_list, _shifted_wave_list = [], []
        _sr = 32_000; _max_sec = 60
        _duration = _sr * _max_sec
        _shift = int(_sr * 2.5)
        for _p in _oof_paths:
            with soundfile.SoundFile(_p) as _f:
                _nfr = _f.frames
                _wave = np.zeros(_shift + _duration + _shift, dtype="float32")
                if _nfr < _duration + _shift:
                    _wave[_shift:_shift + _nfr] = _f.read(dtype="float32")
                else:
                    _wave[_shift:_shift + _duration + _shift] = _f.read(frames=_duration + _shift, dtype="float32")
            for _ss in range(0, _max_sec, 5):
                _wave_list.append(_wave[_shift + _ss * _sr: _shift + (_ss + 5) * _sr])
            for _ss in range(0, _max_sec + 5, 5):
                _shifted_wave_list.append(_wave[_ss * _sr: (_ss + 5) * _sr])

        _bs = 12
        _wave_batches = [np.stack(_wave_list[i:i + _bs], axis=0)
                         for i in range(0, len(_wave_list), _bs)]
        _shifted_batches = [np.stack(_shifted_wave_list[i:i + _bs], axis=0)
                            for i in range(0, len(_shifted_wave_list), _bs)]

        _lms = Parallel(n_jobs=4, verbose=0)(
            delayed(lms_transform)(torch.from_numpy(w)) for w in _wave_batches)
        _shifted_lms = Parallel(n_jobs=4, verbose=0)(
            delayed(lms_transform)(torch.from_numpy(w)) for w in _shifted_batches)
        _lms          = [np.ascontiguousarray(x, dtype=np.float32) for x in _lms]
        _shifted_lms  = [np.ascontiguousarray(x, dtype=np.float32) for x in _shifted_lms]
        del _wave_batches, _shifted_batches; gc.collect()

        _n_segs         = _n_oof_files * 12
        _n_shifted_segs = _n_oof_files * 13
        _preds  = np.zeros((len(ov_model_list), _n_segs,         N_CLASSES), dtype=np.float32)
        _spreds = np.zeros((len(ov_model_list), _n_shifted_segs, N_CLASSES), dtype=np.float32)
        for _mid, _ov in enumerate(ov_model_list):
            _preds[_mid]  = async_infer_with_order(_ov, _lms,         16)
            _spreds[_mid] = async_infer_with_order(_ov, _shifted_lms, 16)

        if RANK_AVG:
            for _mid in range(len(ov_model_list)):
                _preds[_mid]  = rank_normalize(_preds[_mid])
                _spreds[_mid] = rank_normalize(_spreds[_mid])

        # reshape and shift-blend per recording
        _by_rec  = _preds.reshape(N_FOLDS, _n_oof_files, 12, N_CLASSES)
        _sby_rec = _spreds.reshape(N_FOLDS, _n_oof_files, 13, N_CLASSES)
        _tta_by_rec = (
            0.25 * _sby_rec[..., 0:12, :] +
            0.50 * _by_rec +
            0.25 * _sby_rec[..., 1:13, :]
        )
        _hgnet_oof_arr = _tta_by_rec.reshape(N_FOLDS, _n_segs, N_CLASSES).mean(axis=0)

        # Build row_ids in the same scheme as meta_tr.
        _oof_rows = []
        for _p in _oof_paths:
            _stem = _p.stem
            for _e in range(5, 65, 5):
                _oof_rows.append(f"{_stem}_{_e}")
        HGNET_OOF_DF = pd.DataFrame(np.clip(_hgnet_oof_arr, 0.0, 1.0),
                                    columns=PRIMARY_LABELS)
        HGNET_OOF_DF.insert(0, "row_id", _oof_rows)
        print(f"[OOF/HGNet] done in {_time_mod.time() - _t0:.1f}s  shape={HGNET_OOF_DF.shape}")
    except Exception as _exc:
        print(f"[OOF/HGNet] disabled: {_exc}")
        HGNET_OOF_DF = None
else:
    print("[OOF/HGNet] skipped (formal rerun mode)")


In [ ]:
# Complete OOF, part 2: distilled SED on labeled soundscape files.
# Same pipeline as the test-side SED pass (5 ONNX folds, clip + frame-max
# heads averaged, Gaussian smoothing per recording). Gated on IS_DRY_RUN.

import time as _time_mod

import numpy as np
import pandas as pd

SED_OOF_DF = None

if globals().get("IS_DRY_RUN", False):
    try:
        _t0 = _time_mod.time()
        from scipy.ndimage import gaussian_filter1d

        _oof_fnames = sorted(sc["filename"].unique().tolist())
        _oof_paths  = [TRAIN_SS / fn for fn in _oof_fnames if (TRAIN_SS / fn).exists()]
        if len(_oof_paths) == 0:
            raise RuntimeError("no labeled soundscape files found")
        print(f"[OOF/SED] running on {len(_oof_paths)} files across {len(sed_sessions)} folds...")

        _rows, _preds = [], []
        for _i, _p in enumerate(_oof_paths, 1):
            _chunks, _ends = file_to_sed_chunks(_p)
            _mel = audio_to_mel(_chunks)
            _psum = np.zeros((len(_chunks), N_CLASSES), dtype=np.float32)
            for _sess in sed_sessions:
                _outs = _sess.run(None, {_sess.get_inputs()[0].name: _mel})
                _clip_logits = _outs[0]
                _frame_max   = _outs[1].max(axis=1)
                _psum += 0.5 * sigmoid_sed(_clip_logits) + 0.5 * sigmoid_sed(_frame_max)
            _pmean = _psum / len(sed_sessions)
            if len(_pmean) > 1:
                _pmean = gaussian_filter1d(_pmean, sigma=0.65, axis=0,
                                            mode="nearest").astype(np.float32)
            _stem = _p.stem
            _rows.extend([f"{_stem}_{int(_e)}" for _e in _ends])
            _preds.append(_pmean)
            if _i == 1 or _i % 25 == 0 or _i == len(_oof_paths):
                print(f"[OOF/SED] {_i}/{len(_oof_paths)} | {_time_mod.time() - _t0:.1f}s")

        _sed_arr = np.concatenate(_preds, axis=0)
        SED_OOF_DF = pd.DataFrame(np.clip(_sed_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
        SED_OOF_DF.insert(0, "row_id", _rows)
        print(f"[OOF/SED] done in {_time_mod.time() - _t0:.1f}s  shape={SED_OOF_DF.shape}")
    except Exception as _exc:
        print(f"[OOF/SED] disabled: {_exc}")
        SED_OOF_DF = None
else:
    print("[OOF/SED] skipped (formal rerun mode)")


In [ ]:
# Complete OOF, part 3: HTP K-fold + full-blend assembly + macro_auc.
#
# Combines the four OOF sources (ProtoSSM+MLP first-pass from V33's
# run_pipeline_oof; SED on labeled soundscapes; HGNet on labeled
# soundscapes; HTP head K-fold on cached embeddings) into an approximation
# of the V41 final-blend pipeline, and reports macro_auc at each stage.
# The approximation skips the dual-gate rescue rules and the train-audio
# HEAD so the absolute number will undershoot the LB by a small constant;
# what matters for gating future iterations is the *delta* between
# configurations.

import math
import time as _time_mod

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

if globals().get("IS_DRY_RUN", False):
    try:
        _t0 = _time_mod.time()

        # ── HTP K-fold OOF on cached embeddings ────────────────────────
        _groups = meta_tr["filename"].to_numpy()
        _gkf = GroupKFold(n_splits=min(4, len(np.unique(_groups))))
        _htp_oof = np.zeros((emb_tr.shape[0], N_CLASSES), dtype=np.float32)
        for _f, (_tr_i, _va_i) in enumerate(_gkf.split(emb_tr, groups=_groups), 1):
            torch.manual_seed(0 + _f)
            _m = HTPHead(emb_tr.shape[1], HTP_DIM, HTP_PROTOTYPES, HTP_TAU)
            _opt = torch.optim.AdamW(_m.parameters(), lr=3e-3, weight_decay=1e-5)
            _pos = Y_FULL_aligned[_tr_i].sum(axis=0).clip(min=1.0)
            _neg = len(_tr_i) - _pos
            _pw  = torch.tensor((_neg / _pos).clip(1.0, 50.0), dtype=torch.float32)
            _Xt  = torch.tensor(emb_tr[_tr_i], dtype=torch.float32)
            _Yt  = torch.tensor(Y_FULL_aligned[_tr_i], dtype=torch.float32)
            _n   = _Xt.shape[0]
            _rng = np.random.RandomState(_f)
            for _ep in range(15):
                _cos = _ep / 14.0
                _lr  = 1e-4 + 0.5 * (3e-3 - 1e-4) * (1.0 + math.cos(math.pi * _cos))
                for _g in _opt.param_groups: _g["lr"] = _lr
                _perm = _rng.permutation(_n)
                for _i in range(0, _n, 4096):
                    _idx = _perm[_i:_i + 4096]
                    _xb  = _Xt[_idx] * (torch.rand_like(_Xt[_idx]) > 0.1).float()
                    _loss = torch.nn.functional.binary_cross_entropy_with_logits(
                        _m(_xb), _Yt[_idx], pos_weight=_pw)
                    _opt.zero_grad(set_to_none=True); _loss.backward()
                    torch.nn.utils.clip_grad_norm_(_m.parameters(), 5.0); _opt.step()
            _m.eval()
            with torch.no_grad():
                _xv = torch.tensor(emb_tr[_va_i], dtype=torch.float32)
                _lg = np.concatenate([
                    _m(_xv[_i:_i + 8192]).cpu().numpy()
                    for _i in range(0, _xv.shape[0], 8192)
                ], axis=0).astype(np.float32)
            _htp_oof[_va_i] = 1.0 / (1.0 + np.exp(-np.clip(_lg, -30.0, 30.0)))

        # ── ProtoSSM + MLP first-pass OOF (from V33's run_pipeline_oof) ──
        _pp_oof = globals().get("oof_pipeline", None)
        if _pp_oof is None:
            raise RuntimeError("oof_pipeline not in scope; run_pipeline_oof did not run")

        # ── Align HGNet/SED OOF DFs to meta_tr row_id order ────────────
        def _align_df_to_meta(_df, _meta):
            if _df is None or "row_id" not in _df.columns:
                return None
            _by_id = _df.set_index("row_id")
            _need  = _meta["row_id"].astype(str).tolist()
            _miss  = [r for r in _need if r not in _by_id.index]
            if len(_miss) > 0:
                print(f"[OOF/align] {len(_miss)} meta_tr row_ids missing from OOF df; using NaN-fill")
                return None
            return _by_id.loc[_need][PRIMARY_LABELS].to_numpy(np.float32)

        _hgnet_arr = _align_df_to_meta(globals().get("HGNET_OOF_DF"), meta_tr)
        _sed_arr   = _align_df_to_meta(globals().get("SED_OOF_DF"),   meta_tr)

        # ── Rank-normalise + assemble approximation of V41 blend ──────
        def _rank_cols(_x):
            _o = np.empty_like(_x, dtype=np.float32)
            for _j in range(_x.shape[1]):
                _o[:, _j] = pd.Series(_x[:, _j]).rank(method="average", pct=True).to_numpy(np.float32)
            return _o

        _Y = Y_FULL_aligned

        print()
        print("[OOF/report] per-branch macro-AUC on labeled soundscapes:")
        _ppm = macro_auc(_Y, _pp_oof)
        print(f"  ProtoSSM+MLP first-pass : {_ppm:.6f}")
        _hpm = macro_auc(_Y, _htp_oof)
        print(f"  HTP (K-fold)            : {_hpm:.6f}")
        if _sed_arr is not None:
            _sdm = macro_auc(_Y, _sed_arr)
            print(f"  SED (5 folds)           : {_sdm:.6f}")
        if _hgnet_arr is not None:
            _hgm = macro_auc(_Y, _hgnet_arr)
            print(f"  HGNet (5 folds, +-2.5s) : {_hgm:.6f}")

        # pc010 approximation: 0.7 * ProtoSSM+MLP_rank + 0.3 * SED_rank
        if _sed_arr is not None:
            _pc010_r = 0.7 * _rank_cols(np.clip(_pp_oof, 1e-6, 1 - 1e-6)) \
                     + 0.3 * _rank_cols(np.clip(_sed_arr, 1e-6, 1 - 1e-6))
            _pcm = macro_auc(_Y, _pc010_r)
            print(f"  pc010 approx (0.7P+0.3S): {_pcm:.6f}  <-- approximates submission_pc010_raw")
        else:
            _pc010_r = _rank_cols(np.clip(_pp_oof, 1e-6, 1 - 1e-6))
            _pcm = _ppm

        # + HGNet at 0.92/0.08
        if _hgnet_arr is not None:
            _hgnet_r = _rank_cols(np.clip(_hgnet_arr, 1e-6, 1 - 1e-6))
            _blend_r = 0.92 * _pc010_r + 0.08 * _hgnet_r
            _bm = macro_auc(_Y, _blend_r)
            print(f"  pc010_approx+HGNet      : {_bm:.6f}  <-- approximates submission.csv before HTP")
        else:
            _blend_r = _pc010_r
            _bm = _pcm

        # + HTP post-blend at 0.15
        _htp_r = _rank_cols(np.clip(_htp_oof, 1e-6, 1 - 1e-6))
        _full_r = 0.85 * _rank_cols(_blend_r) + 0.15 * _htp_r
        _fm = macro_auc(_Y, _full_r)
        print(f"  FULL approx (+ HTP@0.15): {_fm:.6f}  <-- approximates V41 LB number (0.944)")

        # Sweep the HTP weight against the full blend for the OOF-optimal value.
        print()
        print("[OOF/report] HTP post-blend weight sweep on full blend:")
        _best_w, _best_auc = 0.0, -1.0
        for _w in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
            _b = (1.0 - _w) * _rank_cols(_blend_r) + _w * _htp_r
            _a = macro_auc(_Y, _b)
            _mark = "  *" if _a > _best_auc else ""
            if _a > _best_auc:
                _best_w, _best_auc = _w, _a
            print(f"        w_htp={_w:.2f}  AUC={_a:.6f}{_mark}")
        print(f"[OOF/report] OOF-optimal HTP weight = {_best_w:.2f}  (AUC={_best_auc:.6f}); "
              f"current submission uses 0.15")
        print()
        print(f"[OOF/report] done in {_time_mod.time() - _t0:.1f}s")
    except Exception as _exc:
        print(f"[OOF/report] disabled: {_exc}")
else:
    print("[OOF/report] skipped (formal rerun mode)")
